# ConvNeXt Stem → Vision Mamba Encoder → CrossScan Bottleneck → Mamba Decoder
## LoveDA Semantic Segmentation

### Architecture
```
Input (B, 3, 512, 512)
│
├─ ① ConvNeXt-Tiny Stem  (pretrained ImageNet-1K, ~28M)
│      C0: (B,  96, 128, 128)  stride  4
│      C1: (B, 192,  64,  64)  stride  8
│      C2: (B, 384,  32,  32)  stride 16
│      C3: (B, 768,  16,  16)  stride 32
│
├─ ② CNN → Mamba Bridge  (DW-Conv + 1×1 → embed_dim=96 per scale)
│      4 × (B, Hi*Wi, 96)  token sequences
│
├─ ③ Vision Mamba Encoder  ← NEW: SS2D at every scale
│      Scale 0,1: WindowedSS2DLayer   (local window SS2D + shifted + global SS2D)
│      Scale 2,3: SS2DLayer           (full-resolution 4-direction SS2D)
│      Each layer: N × VSSBlock  (SS2D + FFN)
│      Saves skip tokens at scales 0,1,2
│
├─ ④ CrossScan Mamba Bottleneck  (row BiMamba + col BiMamba → cross-fuse)
│      Auxiliary seg head (deep supervision ×0.4)
│
├─ ⑤ Mamba Decoder  (PatchExpand + concat skip + MambaBlock) × 3
│
└─ ⑥ Final Head  (ConvTranspose ×2 → Conv 1×1) → (B, 7, 512, 512)
```

### What changed vs the ConvNeXt+Mamba notebook
| Component | Before | Now |
|---|---|---|
| Encoder (shallow scales 0,1) | `WindowedLocalMambaLayer` (1-D window + global Mamba) | **`WindowedSS2DLayer`** (4-dir SS2D inside windows + shifted windows + global SS2D) |
| Encoder (deep scales 2,3)   | `BidirectionalMambaLayer` (1-D fwd+bwd) | **`SS2DLayer`** (4-direction SS2D: →←↓↑) |
| Encoder block structure | Single Mamba op per scale | **VSSBlock** (SS2D + LayerNorm + FFN) × depth |
| 2-D spatial awareness | Only via window partitioning | **True 2-D via 4-direction scan at every scale** |

Everything else unchanged: ConvNeXt stem, bridge, CrossScan bottleneck,
Mamba decoder, concat skips, deep supervision, CutMix, class weights, grad clipping.


## 1. Environment

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✓ CUDA environment configured')

In [ ]:
# ── Cell 3: kept for reference — actual install is in Cell 5 ──────────────
# Do NOT run this cell on Kaggle — it reinstalls PyTorch unnecessarily
# and wipes the correct CUDA toolkit. Run Cell 5 instead.


In [ ]:
# Step 1: Find your environment
!python --version
!python -c "import torch; print(torch.__version__, torch.version.cuda)"

In [ ]:
!pip install ninja
!MAX_JOBS=1 pip install causal-conv1d mamba-ssm --no-cache-dir --no-build-isolation --no-binary :all:

In [ ]:
# !pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
#     --index-url https://download.pytorch.org/whl/cu124

In [ ]:
import torch
print(torch.__version__)        # should be 2.5.1
print(torch.version.cuda)       # should be 12.4
print(torch.cuda.is_available()) # should be True

In [ ]:
!pip install causal-conv1d mamba-ssm

In [ ]:
# !pip install causal-conv1d==1.6.0 mamba-ssm==2.2.3.post2

In [ ]:
# !pip install ninja packaging
# !pip install git+https://github.com/Dao-AILab/causal-conv1d.git
# !pip install git+https://github.com/state-spaces/mamba.git

## 2. Install

In [ ]:
# import sys, subprocess, torch

# def run(cmd):
#     r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     if r.returncode != 0:
#         raise RuntimeError(r.stderr[-600:])

# print(f"Python=cp{sys.version_info.major}{sys.version_info.minor} | "
#       f"torch={torch.__version__} | CUDA={torch.version.cuda}")
# assert torch.cuda.is_available(), "⚠ Enable GPU: Settings → Accelerator → GPU T4/P100"

# # ── Remove any mismatched wheels already on disk ──────────────────────────────
# print("① Removing old causal-conv1d / mamba-ssm...")
# run("pip uninstall causal-conv1d mamba-ssm -y")

# # ── causal-conv1d 1.2.2 + mamba-ssm 1.2.2 — known-good pair ─────────────────
# #    causal-conv1d >= 1.4.0 changed causal_conv1d_fwd() signature (added seq_idx
# #    as 8th arg). mamba-ssm 1.2.2 calls it with 7 args → TypeError at runtime.
# #    Pinning both to 1.2.2 ensures the call signature matches exactly.
# print("② Building causal-conv1d==1.2.2 (~4 min)...")
# print("① causal-conv1d==1.2.2.post1...")
# run("MAX_JOBS=4 pip install causal-conv1d==1.2.2.post1 "
#     "--no-cache-dir --no-build-isolation --no-binary :all: -q")
# print("   ✓")

# print("③ Building mamba-ssm==1.2.2 (~6 min)...")
# run("MAX_JOBS=4 pip install mamba-ssm==1.2.2 "
#     "--no-cache-dir --no-build-isolation --no-binary :all: -q")
# print("   ✓ mamba-ssm==1.2.2")

# # ── transformers pin — GreedySearchDecoderOnlyOutput removed in >= 4.41 ───────
# print("④ Pinning transformers==4.40.0...")
# run("pip install transformers==4.40.0 -q")
# print("   ✓ transformers==4.40.0")

# print("⑤ timm / einops / albumentations...")
# run("pip install timm einops albumentations -q")
# print("   ✓")

# # ── Restart so the newly installed packages are picked up cleanly ─────────────
# print("\n⚡ Restarting kernel...")
# import IPython
# IPython.Application.instance().kernel.do_shutdown(restart=True)


In [ ]:
# import torch
# from mamba_ssm import Mamba
# import timm, transformers, causal_conv1d, mamba_ssm

# # Quick functional smoke test for Mamba itself
# _m = Mamba(d_model=96, d_state=16, d_conv=4, expand=2).cuda()
# _x = torch.randn(2, 64, 96).cuda()
# _o = _m(_x)
# assert _o.shape == _x.shape, "Mamba output shape mismatch"
# del _m, _x, _o

# print("✅ All packages verified")
# print(f"   torch         {torch.__version__} | {torch.cuda.get_device_name(0)}")
# print(f"   causal-conv1d {causal_conv1d.__version__}")
# print(f"   mamba-ssm     {mamba_ssm.__version__}")
# print(f"   transformers  {transformers.__version__}")
# print(f"   timm          {timm.__version__}")


In [ ]:
# ── Old install commands — superseded by Cell 5 above. Do not run. ──────────


In [ ]:
#extra- Step 1 — check what torch and CUDA version Kaggle gave us
!python -c "import torch; print('torch:', torch.__version__, '| cuda:', torch.version.cuda)"

# Step 2 — install causal-conv1d first (mamba-ssm depends on it)
# --no-build-isolation makes it use Kaggle's existing PyTorch + CUDA
!pip install causal-conv1d --no-build-isolation -q

# Step 3 — install mamba-ssm using the same flag
!pip install mamba-ssm --no-build-isolation -q

# Step 4 — install other dependencies
!pip install timm einops albumentations -q

print('✓ All installed')

## 3. Imports

In [ ]:
import os, random, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import timm
from mamba_ssm import Mamba
from einops import rearrange
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 4. Config

In [ ]:
class Config:
    # ── Dataset ──────────────────────────────────────────────────────────
    DATASET_BASE = '/kaggle/input/datasets/mohammedjaveed/loveda-dataset'
    TRAIN_PATH = VAL_PATH = TEST_PATH = None
    if os.path.exists(DATASET_BASE):
        for _split, _attr in [('Train','TRAIN_PATH'),('Val','VAL_PATH'),('Test','TEST_PATH')]:
            for _p in [os.path.join(DATASET_BASE, _split, _split),
                       os.path.join(DATASET_BASE, _split)]:
                if os.path.exists(_p) and os.path.exists(os.path.join(_p,'Urban')):
                    locals()[_attr] = _p; break
    OUTPUT_DIR  = './outputs'

    # ── Image ─────────────────────────────────────────────────────────────
    IMG_SIZE    = 512
    IN_CHANS    = 3
    NUM_CLASSES = 7

    # ── ConvNeXt Backbone ─────────────────────────────────────────────────
    BACKBONE            = 'convnext_tiny'   # convnext_small / convnext_base
    BACKBONE_PRETRAINED = True
    FREEZE_EPOCHS       = 3                 # ↓ was 5 — fewer warmup epochs

    # ── Vision Mamba Encoder ──────────────────────────────────────────────
    EMBED_DIM   = 96      # unified dim after CNN bridge
    DEPTHS      = [2, 3, 4, 2]             # ↓ was [2,4,6,2] — fewer VSSBlocks (faster)
    D_STATE     = 12
    D_CONV      = 4
    EXPAND      = 2
    WINDOW_SIZE = 8       # spatial window size for windowed SS2D
    DROPOUT     = 0.2

    # ── Training ──────────────────────────────────────────────────────────
    EPOCHS        = 20                    # ↓ was 100 — early stop handles rest
    BATCH_SIZE    = 8
    LR_BACKBONE   = 6e-5
    LR_DECODER    = 2e-3
    WEIGHT_DECAY  = 1e-2
    NUM_WORKERS   = 4
    SAVE_FREQ     = 10                     # periodic checkpoint every N epochs

    # ── Early Stopping ────────────────────────────────────────────────────
    PATIENCE      = 12                     # stop if no improvement for N epochs

    # ── Gradient Accumulation ─────────────────────────────────────────────
    ACCUM_STEPS   = 2                      # effective batch = BATCH_SIZE × ACCUM_STEPS

    # ── CutMix ───────────────────────────────────────────────────────────
    CUTMIX_PROB  = 0.5
    CUTMIX_ALPHA = 1.0

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
print(f'Train     : {config.TRAIN_PATH}')
print(f'Val       : {config.VAL_PATH}')
print(f'Backbone  : {config.BACKBONE}')
print(f'Embed dim : {config.EMBED_DIM}')
print(f'Depths    : {config.DEPTHS}')
print(f'Patience  : {config.PATIENCE}')
print(f'Accum     : {config.ACCUM_STEPS}x  (eff. batch={config.BATCH_SIZE*config.ACCUM_STEPS})')


## 5. ConvNeXt Stem
*(unchanged — pretrained ImageNet feature extractor)*

In [ ]:
class ConvNeXtStem(nn.Module):
    """
    Pretrained ConvNeXt-Tiny via timm. Returns 4 feature maps at strides 4→32.
    ConvNeXt-Tiny dims: [96, 192, 384, 768]
    """
    def __init__(self, name='convnext_tiny', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            name, pretrained=pretrained,
            features_only=True, out_indices=(0,1,2,3))
        self.feat_dims = self.backbone.feature_info.channels()
        print(f'ConvNeXt : {name}  pretrained={pretrained}')
        print(f'Dims     : {self.feat_dims}  (strides 4/8/16/32)')

    def forward(self, x): return self.backbone(x)

    def freeze(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print('ConvNeXt frozen')

    def unfreeze(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        print('ConvNeXt unfrozen')

print('✓ ConvNeXtStem defined')

## 6. CNN → Mamba Bridge
*(unchanged)*

In [ ]:
class ConvNeXtMambaBridge(nn.Module):
    """
    Per-scale: DW-Conv 3×3 → BN → GELU → PW-Conv 1×1 → BN → GELU
                → Dropout2d → flatten → LayerNorm → (B, H*W, embed_dim)
    """
    def __init__(self, cnn_dims, embed_dim, dropout=0.1):
        super().__init__()
        self.projs = nn.ModuleList()
        for cin in cnn_dims:
            self.projs.append(nn.Sequential(
                nn.Conv2d(cin, cin,      3, padding=1, groups=cin, bias=False),
                nn.BatchNorm2d(cin), nn.GELU(),
                nn.Conv2d(cin, embed_dim, 1, bias=False),
                nn.BatchNorm2d(embed_dim), nn.GELU(),
                nn.Dropout2d(dropout),
            ))
        self.norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in cnn_dims])

    def forward(self, cnn_feats):
        tokens, sizes = [], []
        for i, feat in enumerate(cnn_feats):
            x = self.projs[i](feat)
            B, D, H, W = x.shape
            sizes.append((H, W))
            x = x.flatten(2).transpose(1, 2)
            tokens.append(self.norms[i](x))
        return tokens, sizes

print('✓ ConvNeXtMambaBridge defined')

## 7. Vision Mamba Core: SS2D + VSSBlock  ← NEW

### SS2D (2D Selective Scan)
The core Vision Mamba primitive. Scans the spatial feature map in **4 directions**:

```
→  right  :  row-major forward     (left→right, top→bottom)
←  left   :  row-major backward    (right→left, bottom→top)
↓  down   :  col-major forward     (top→bottom, left→right)
↑  up     :  col-major backward    (bottom→top, right→left)
```

Each direction has its own independent Mamba SSM. The 4 outputs are
merged via a learned `Linear(4C → C)` projection.

This is fundamentally different from the old `WindowedLocalMambaLayer` /
`BidirectionalMambaLayer` which only scanned in 1-D (left↔right).
SS2D gives every token full 2-D spatial context.

### VSSBlock
`SS2D` + `LayerNorm` + `FFN` — the standard Vision Mamba block.
Stacked `N` times per encoder scale.

In [ ]:
class SS2D(nn.Module):
    """
    2-D Selective Scan — the core Vision Mamba primitive.

    Scans feature map in 4 spatial directions (→ ← ↓ ↑), each with its
    own independent Mamba SSM. The 4 outputs are merged by a learned
    linear projection, giving every token full 2-D spatial context.

    Unlike plain bidirectional Mamba (which only scans left↔right),
    SS2D captures both horizontal AND vertical spatial dependencies.
    """
    def __init__(self, dim, d_state=16, d_conv=4, expand=2, dropout=0.0):
        super().__init__()
        self.dim      = dim
        self.norm     = nn.LayerNorm(dim)
        # Split input into 4 branches before scanning
        self.in_proj  = nn.Linear(dim, dim * 4, bias=False)
        # One independent Mamba SSM per scan direction
        self.ssm_r    = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)  # →
        self.ssm_l    = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)  # ←
        self.ssm_d    = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)  # ↓
        self.ssm_u    = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)  # ↑
        # Merge 4 direction outputs → dim
        self.out_proj = nn.Linear(dim * 4, dim, bias=False)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, H, W, C)  — spatial format
        B, H, W, C = x.shape
        xn = self.norm(x)                                       # pre-norm
        branches = self.in_proj(xn).chunk(4, dim=-1)           # 4 × (B,H,W,C)

        # ── → right scan  (standard row-major) ───────────────────────────
        y_r = self.ssm_r(branches[0].reshape(B, H*W, C))

        # ── ← left scan   (reversed row-major) ───────────────────────────
        seq_l = torch.flip(branches[1].reshape(B, H*W, C), [1])
        y_l   = torch.flip(self.ssm_l(seq_l), [1])

        # ── ↓ down scan   (col-major: transpose H↔W, scan, transpose back) ─
        seq_d = branches[2].permute(0, 2, 1, 3).contiguous().reshape(B, H*W, C)
        y_d   = self.ssm_d(seq_d).reshape(B, W, H, C).permute(0, 2, 1, 3).contiguous().reshape(B, H*W, C)

        # ── ↑ up scan     (reversed col-major) ───────────────────────────
        seq_u = torch.flip(
            branches[3].permute(0, 2, 1, 3).contiguous().reshape(B, H*W, C), [1])
        y_u   = torch.flip(self.ssm_u(seq_u), [1]).reshape(B, W, H, C).permute(0, 2, 1, 3).contiguous().reshape(B, H*W, C)

        # Merge all 4 directions and project back to dim
        merged = torch.cat([y_r, y_l, y_d, y_u], dim=-1)       # (B, H*W, 4C)
        out    = self.out_proj(merged).reshape(B, H, W, C)      # (B, H,  W,  C)
        return x + self.drop(out)                               # residual


class VSSBlock(nn.Module):
    """
    Vision State Space Block = SS2D + FFN with pre-norm.
    Standard building block of the Vision Mamba encoder.

    Input/output: (B, H, W, C)  — spatial tensor format
    """
    def __init__(self, dim, d_state=16, d_conv=4, expand=2,
                 mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.ss2d   = SS2D(dim, d_state, d_conv, expand, dropout)
        self.norm   = nn.LayerNorm(dim)
        hidden      = int(dim * mlp_ratio)
        self.ffn    = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: (B, H, W, C)
        x = self.ss2d(x)                  # 4-direction SS2D + residual
        x = x + self.ffn(self.norm(x))    # FFN + residual
        return x

print('✓ SS2D and VSSBlock defined')

## 8. Windowed SS2D Encoder Layer (Shallow Scales 0, 1)

For high-resolution scales (stride 4 = 128×128 = 16,384 tokens), running
full SS2D over the entire sequence is expensive. This layer applies SS2D
inside local spatial windows — the same window strategy as Swin Transformer,
but with SS2D scanning instead of self-attention.

Three components per layer:
1. **Normal-Window VSSBlock**: SS2D inside W×W non-overlapping windows
2. **Shifted-Window VSSBlock**: Same but shifted by W/2 — bridges boundaries
3. **Global VSSBlock**: Full-sequence SS2D for long-range context (after windows compress it)

In [ ]:
class WindowedSS2DLayer(nn.Module):
    """
    Windowed Vision Mamba layer for high-resolution scales.

    Applies SS2D inside local windows to manage token count, then
    runs a global SS2D pass for cross-window long-range context.

    Structure:
      [Normal window SS2D] → [Shifted window SS2D] → [Global SS2D]

    The window + shifted-window pair is the same design as Swin-T,
    but replaces multi-head attention with SS2D (4-direction scan).
    """
    def __init__(self, dim, window_size=8, depth=2,
                 d_state=16, d_conv=4, expand=2, dropout=0.0):
        super().__init__()
        self.window_size = window_size

        # Normal-window VSSBlocks (no shift)
        self.normal_blocks = nn.ModuleList([
            VSSBlock(dim, d_state, d_conv, expand, dropout=dropout)
            for _ in range(max(depth // 2, 1))
        ])
        # Shifted-window VSSBlocks (shift = window_size // 2)
        self.shifted_blocks = nn.ModuleList([
            VSSBlock(dim, d_state, d_conv, expand, dropout=dropout)
            for _ in range(max(depth // 2, 1))
        ])
        # Global SS2D: one VSSBlock on full-res tokens for cross-window context
        self.global_block = VSSBlock(dim, d_state, d_conv, expand, dropout=dropout)

    def _partition(self, x):
        """Partition (B,H,W,C) into non-overlapping windows of window_size×window_size."""
        B, H, W, C = x.shape
        ws = self.window_size
        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        if pad_h or pad_w:
            x = F.pad(x.permute(0,3,1,2), (0,pad_w,0,pad_h)).permute(0,2,3,1)
        Hp, Wp = H + pad_h, W + pad_w
        x = x.view(B, Hp//ws, ws, Wp//ws, ws, C)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, ws, ws, C)
        return x, (B, H, W, Hp, Wp)

    def _unpartition(self, x, info):
        """Reverse window partition back to (B, H, W, C)."""
        B, H, W, Hp, Wp = info
        ws = self.window_size
        x  = x.view(B, Hp//ws, Wp//ws, ws, ws, -1)
        x  = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, Hp, Wp, -1)
        return x[:, :H, :W, :].contiguous()

    def forward(self, x, H, W):
        # x: (B, H*W, C)  — token sequence format from bridge
        B, L, C = x.shape
        x2d = x.view(B, H, W, C)          # → spatial format for SS2D

        # ── Normal-window pass ────────────────────────────────────────────
        wins, info = self._partition(x2d)
        for blk in self.normal_blocks:
            wins = blk(wins)
        x2d = self._unpartition(wins, info)

        # ── Shifted-window pass ───────────────────────────────────────────
        shift = self.window_size // 2
        x2d_shifted = torch.roll(x2d, shifts=(-shift, -shift), dims=(1, 2))
        wins_s, info_s = self._partition(x2d_shifted)
        for blk in self.shifted_blocks:
            wins_s = blk(wins_s)
        x2d = torch.roll(self._unpartition(wins_s, info_s),
                         shifts=(shift, shift), dims=(1, 2))

        # ── Global SS2D pass ──────────────────────────────────────────────
        x2d = self.global_block(x2d)

        return x2d.view(B, H*W, C)        # → back to token sequence format

print('✓ WindowedSS2DLayer defined')

## 9. Full SS2D Encoder Layer (Deep Scales 2, 3)

For deep scales (stride 16 = 32×32 = 1,024 tokens, stride 32 = 16×16 = 256 tokens)
the sequence is short enough to run SS2D directly over the full spatial map.
No windowing needed — full 2-D context at every token.

In [ ]:
class SS2DLayer(nn.Module):
    """
    Full-resolution Vision Mamba layer for deep (low-resolution) scales.

    Runs N × VSSBlock directly on the full spatial feature map.
    No windowing — at stride 16/32 the map is small enough for
    full SS2D to be efficient (256–1024 tokens).

    Input/output: (B, H*W, C) token sequence (converted internally to spatial)
    """
    def __init__(self, dim, depth=2, d_state=16, d_conv=4,
                 expand=2, dropout=0.0):
        super().__init__()
        self.blocks = nn.ModuleList([
            VSSBlock(dim, d_state, d_conv, expand, dropout=dropout)
            for _ in range(depth)
        ])

    def forward(self, x, H, W):
        # x: (B, H*W, C) — convert to spatial, apply SS2D blocks, convert back
        B, L, C = x.shape
        x2d = x.view(B, H, W, C)          # → (B, H, W, C) for SS2D
        for blk in self.blocks:
            x2d = blk(x2d)                 # each block: SS2D + FFN
        return x2d.view(B, H*W, C)         # → token sequence

print('✓ SS2DLayer defined')

## 10. Remaining Mamba Blocks
*(Bottleneck BiMamba + Decoder MambaBlock — unchanged)*

In [ ]:
class BidirectionalMambaLayer(nn.Module):
    """Gated bidirectional 1-D Mamba — used in CrossScan Bottleneck and Decoder."""
    def __init__(self, dim, d_state=16, d_conv=4, expand=2, dropout=0.0):
        super().__init__()
        self.norm      = nn.LayerNorm(dim)
        self.mamba_fwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_bwd = Mamba(d_model=dim, d_state=d_state, d_conv=d_conv, expand=expand)
        self.gate      = nn.Linear(dim, dim, bias=True)
        self.drop      = nn.Dropout(dropout)

    def forward(self, x):
        res = x; xn = self.norm(x)
        fwd = self.mamba_fwd(xn)
        bwd = torch.flip(self.mamba_bwd(torch.flip(xn, [1])), [1])
        g   = torch.sigmoid(self.gate(fwd + bwd))
        return res + self.drop(g * fwd + (1 - g) * bwd)


class MambaBlock(nn.Module):
    """Stack of BidirMamba layers — used in Decoder."""
    def __init__(self, dim, depth=2, d_state=16, d_conv=4, expand=2, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            BidirectionalMambaLayer(dim, d_state, d_conv, expand, dropout)
            for _ in range(depth)
        ])
    def forward(self, x):
        for l in self.layers: x = l(x)
        return x


class CrossScanMambaBottleneck(nn.Module):
    """Row BiMamba + Col BiMamba, cross-fused. Deep compression stage."""
    def __init__(self, dim, depth=4, d_state=16, d_conv=4, expand=2, dropout=0.0):
        super().__init__()
        half = max(depth // 2, 1)
        self.row_layers = nn.ModuleList([
            BidirectionalMambaLayer(dim, d_state, d_conv, expand, dropout)
            for _ in range(half)])
        self.col_layers = nn.ModuleList([
            BidirectionalMambaLayer(dim, d_state, d_conv, expand, dropout)
            for _ in range(half)])
        self.fuse = nn.Linear(dim * 2, dim)

    def forward(self, x):
        B, L, C = x.shape
        H = W = int(L**0.5)
        x_row = x
        for l in self.row_layers: x_row = l(x_row)
        x_col = x.view(B,H,W,C).permute(0,2,1,3).contiguous().view(B,L,C)
        for l in self.col_layers: x_col = l(x_col)
        x_col = x_col.view(B,W,H,C).permute(0,2,1,3).contiguous().view(B,L,C)
        return self.fuse(torch.cat([x_row, x_col], dim=-1))

print('✓ BiMamba + CrossScan Bottleneck + MambaBlock defined')

## 11. ConvNeXtVisionMambaUNet — Full Model

In [ ]:
class ConvNeXtVisionMambaUNet(nn.Module):
    """
    ConvNeXt Stem → Vision Mamba Encoder → CrossScan Bottleneck → Mamba Decoder

    Encoder uses Vision Mamba (SS2D) throughout:
      - Shallow scales (0,1): WindowedSS2DLayer  (windowed + shifted + global SS2D)
      - Deep scales   (2,3): SS2DLayer           (full-resolution SS2D)

    Everything else identical to the ConvNeXt+Mamba notebook:
      - ConvNeXtMambaBridge, CrossScanMambaBottleneck, MambaBlock decoder,
        concat skip connections, deep supervision aux head, PatchExpand upsample.
    """
    def __init__(self,
                 backbone            = 'convnext_tiny',
                 backbone_pretrained = True,
                 num_classes         = 7,
                 embed_dim           = 96,
                 depths              = (2, 4, 6, 2),
                 d_state             = 16,
                 d_conv              = 4,
                 expand              = 2,
                 window_size         = 8,
                 dropout             = 0.1):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim   = embed_dim
        self.num_layers  = len(depths)

        # ── ① ConvNeXt Stem ───────────────────────────────────────────────
        self.stem    = ConvNeXtStem(backbone, backbone_pretrained)
        cnn_dims     = self.stem.feat_dims

        # ── ② CNN → Mamba Bridge ──────────────────────────────────────────
        self.bridge  = ConvNeXtMambaBridge(cnn_dims, embed_dim, dropout)

        # ── ③ Vision Mamba Encoder ─────────────────────────────────────────
        # Shallow scales (0,1) → WindowedSS2DLayer  (high-res: 128×128, 64×64)
        # Deep scales   (2,3) → SS2DLayer           (low-res:   32×32, 16×16)
        self.encoder_layers = nn.ModuleList()
        for i in range(self.num_layers):
            if i < 2:
                self.encoder_layers.append(
                    WindowedSS2DLayer(
                        dim=embed_dim, window_size=window_size,
                        depth=depths[i], d_state=d_state,
                        d_conv=d_conv, expand=expand, dropout=dropout))
            else:
                self.encoder_layers.append(
                    SS2DLayer(
                        dim=embed_dim, depth=depths[i],
                        d_state=d_state, d_conv=d_conv,
                        expand=expand, dropout=dropout))

        # ── ④ CrossScan Bottleneck ─────────────────────────────────────────
        self.bottleneck = CrossScanMambaBottleneck(
            embed_dim, depth=depths[-1],
            d_state=d_state, d_conv=d_conv, expand=expand, dropout=dropout)

        # Deep supervision auxiliary head on bottleneck output
        self.aux_head = nn.Linear(embed_dim, num_classes)

        # ── ⑤ Decoder: 3 upsample stages (scale 3→2→1→0) ──────────────────
        self.upsample_layers = nn.ModuleList()
        self.skip_projs      = nn.ModuleList()
        self.decoder_layers  = nn.ModuleList()

        for i in range(self.num_layers - 1):
            # Pixel-shuffle upsample: Linear(C → 4C)
            self.upsample_layers.append(
                nn.Linear(embed_dim, embed_dim * 4, bias=False))
            # Concat skip + upsampled → embed_dim
            self.skip_projs.append(nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim, bias=False),
                nn.LayerNorm(embed_dim), nn.GELU(),
            ))
            # Bidirectional Mamba decode blocks
            self.decoder_layers.append(
                MambaBlock(embed_dim, depths[self.num_layers-2-i],
                           d_state, d_conv, expand, dropout))

        # ── ⑥ Final Head ──────────────────────────────────────────────────
        self.final_expand = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, embed_dim//2, 2, stride=2),
            nn.BatchNorm2d(embed_dim//2), nn.GELU(),
            nn.ConvTranspose2d(embed_dim//2, embed_dim//4, 2, stride=2),
            nn.BatchNorm2d(embed_dim//4), nn.GELU(),
        )
        self.seg_head = nn.Conv2d(embed_dim//4, num_classes, 1)
        self._init_new_weights()

    def _init_new_weights(self):
        for module in [self.bridge, self.encoder_layers, self.bottleneck,
                       self.aux_head, self.upsample_layers, self.skip_projs,
                       self.decoder_layers, self.final_expand, self.seg_head]:
            for m in (module.modules() if hasattr(module,'modules') else [module]):
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None: nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.LayerNorm, nn.BatchNorm2d)):
                    nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.kaiming_normal_(m.weight, mode='fan_out')
                    if m.bias is not None: nn.init.zeros_(m.bias)

    def backbone_parameters(self):
        return list(self.stem.parameters())

    def new_parameters(self):
        return (list(self.bridge.parameters())        +
                list(self.encoder_layers.parameters()) +
                list(self.bottleneck.parameters())    +
                list(self.aux_head.parameters())      +
                list(self.upsample_layers.parameters())+
                list(self.skip_projs.parameters())    +
                list(self.decoder_layers.parameters()) +
                list(self.final_expand.parameters())  +
                list(self.seg_head.parameters()))

    def forward(self, x):
        B, C, H, W = x.shape

        # ① ConvNeXt stem
        cnn_feats = self.stem(x)

        # ② Bridge → token sequences
        tokens, sizes = self.bridge(cnn_feats)

        # ③ Vision Mamba Encoder — refine each scale, save skips
        skips      = []
        enc_tokens = []
        for i in range(self.num_layers):
            Hi, Wi = sizes[i]
            t = self.encoder_layers[i](tokens[i], Hi, Wi)
            enc_tokens.append(t)
            if i < self.num_layers - 1:
                skips.append((t, Hi, Wi))

        # ④ CrossScan Bottleneck on deepest scale
        H_btl, W_btl = sizes[-1]
        x_dec = self.bottleneck(enc_tokens[-1])

        # Deep supervision auxiliary output (training only)
        aux_out = None
        if self.training:
            aux = self.aux_head(x_dec)
            aux = aux.permute(0,2,1).view(B, self.num_classes, H_btl, W_btl)
            aux_out = F.interpolate(aux, (H,W), mode='bilinear', align_corners=False)

        # ⑤ Decoder — upsample + skip fusion + MambaBlock
        cur_H, cur_W = H_btl, W_btl
        for i in range(self.num_layers - 1):
            B2, L2, C2 = x_dec.shape
            # Pixel-shuffle 2× upsample
            x_exp = self.upsample_layers[i](x_dec)       # (B, L, 4C)
            x_exp = x_exp.view(B2, cur_H, cur_W, 4*C2)
            x_exp = rearrange(x_exp,
                'b h w (p1 p2 c)->b (h p1)(w p2) c', p1=2, p2=2, c=C2)
            cur_H, cur_W = cur_H*2, cur_W*2
            x_exp = x_exp.view(B2, cur_H*cur_W, C2)
            # Concat skip from encoder
            skip_x, _, _ = skips[-(i+1)]
            x_dec = self.skip_projs[i](torch.cat([x_exp, skip_x], dim=-1))
            x_dec = self.decoder_layers[i](x_dec)

        # ⑥ Final head
        x_dec = x_dec.view(B, cur_H, cur_W, self.embed_dim).permute(0,3,1,2)
        out   = self.seg_head(self.final_expand(x_dec))
        if out.shape[2:] != (H, W):
            out = F.interpolate(out, (H,W), mode='bilinear', align_corners=False)

        if self.training and aux_out is not None:
            return out, aux_out
        return out

print('✓ ConvNeXtVisionMambaUNet defined')

## 12. Dataset

In [ ]:
class LoveDADataset(Dataset):
    CLASSES  = ['background','building','road','water','barren','forest','agricultural']
    NUM_CLASSES = 7
    PALETTE  = [[0,0,0],[255,0,0],[255,255,0],[0,0,255],
                [159,129,183],[0,255,0],[255,195,128]]

    def __init__(self, root_path, transform=None, has_masks=True):
        self.transform=transform; self.has_masks=has_masks
        self.image_paths=[]; self.mask_paths=[] if has_masks else None
        print(f'Loading: {root_path}')
        for region in ['Urban','Rural']:
            img_dir  = os.path.join(root_path, region, 'images_png')
            mask_dir = os.path.join(root_path, region, 'masks_png')
            if not os.path.exists(img_dir): continue
            for f in sorted(os.listdir(img_dir)):
                if not f.endswith('.png'): continue
                mp = os.path.join(mask_dir, f)
                if has_masks and not os.path.exists(mp): continue
                self.image_paths.append(os.path.join(img_dir, f))
                if has_masks: self.mask_paths.append(mp)
            print(f'  {region}: {len([f for f in os.listdir(img_dir) if f.endswith(".png")])} imgs')
        print(f'  Total: {len(self.image_paths)}')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.image_paths[idx]).convert('RGB'))
        if self.has_masks:
            mask = np.clip(np.array(Image.open(self.mask_paths[idx])),0,6).astype(np.uint8)
            if self.transform:
                t = self.transform(image=img, mask=mask)
                img, mask = t['image'], t['mask']
            if not isinstance(mask, torch.Tensor): mask = torch.from_numpy(mask)
            return img, mask.long()
        else:
            if self.transform: img = self.transform(image=img)['image']
            return img, os.path.basename(self.image_paths[idx])

    @staticmethod
    def decode_segmap(lm):
        rgb = np.zeros((*lm.shape,3),dtype=np.uint8)
        for lid,c in enumerate(LoveDADataset.PALETTE): rgb[lm==lid]=c
        return rgb

print('✓ LoveDADataset defined')

## 13. Loss Functions
*(CE + Dice + Focal, all fixes preserved)*

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth=smooth
    def forward(self, pred, target):
        p=F.softmax(pred,1); t=F.one_hot(target,pred.shape[1]).permute(0,3,1,2).float()
        inter=(p*t).sum(dim=(2,3)); union=p.sum(dim=(2,3))+t.sum(dim=(2,3))
        return 1-((2*inter+self.smooth)/(union+self.smooth)).mean()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def forward(self, pred, target):
        log_p=F.log_softmax(pred,1)
        pt=torch.exp(log_p).gather(1,target.unsqueeze(1)).squeeze(1)
        return (-(1-pt)**self.gamma * log_p.gather(1,target.unsqueeze(1)).squeeze(1)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, num_classes=7, ce_weight=0.3, dice_weight=0.3,
                 focal_weight=0.4, focal_gamma=2.0,
                 class_weights=None, aux_weight=0.4):
        super().__init__()
        self.num_classes=num_classes
        self.ce_w=ce_weight; self.dice_w=dice_weight
        self.foc_w=focal_weight; self.aux_w=aux_weight
        self.ce=nn.CrossEntropyLoss(weight=class_weights)
        self.dice=DiceLoss(); self.focal=FocalLoss(gamma=focal_gamma)

    def _seg(self, pred, target):
        target=torch.clamp(target,0,self.num_classes-1)
        return self.ce_w*self.ce(pred,target)+self.dice_w*self.dice(pred,target)+self.foc_w*self.focal(pred,target)

    def forward(self, pred, target):
        if isinstance(pred, tuple):
            main, aux = pred
            return self._seg(main,target)+self.aux_w*self._seg(aux,target)
        return self._seg(pred, target)
        
def calculate_miou(pred, target, num_classes=7):
    ious, pred, target = [], pred.view(-1), target.view(-1)
    for c in range(num_classes):
        pc,tc=pred==c,target==c
        inter=(pc&tc).sum().float(); union=(pc|tc).sum().float()
        ious.append(float('nan') if union==0 else (inter/union).item())
    valid=[v for v in ious if not math.isnan(v)]
    return float(np.mean(valid)) if valid else 0.0
#code added-extra
# ─────────────────────────────────────────────────────────────────────────────
# ALL SEGMENTATION METRICS — add these before print('✓ Loss functions defined')
# ─────────────────────────────────────────────────────────────────────────────

def compute_all_metrics(pred, target, num_classes=7):
    """
    Computes every metric from a single pred/target pair (CPU tensors).
    Call this after each val batch and accumulate, OR call on full arrays.

    Metrics returned:
      pa      — Pixel Accuracy        : % of all pixels correctly classified
      mpa     — Mean Pixel Accuracy   : average per-class pixel accuracy
      miou    — Mean IoU              : average Intersection over Union per class
      fwiou   — Frequency-Weighted IoU: IoU weighted by how often each class appears
      iou_per — list of IoU per class  (7 values, one per LoveDA class)
      acc_per — list of Accuracy per class
      prec_per— list of Precision per class
      rec_per — list of Recall per class
      f1_per  — list of F1 per class
    """
    pf    = pred.reshape(-1)       # flatten prediction  → 1D
    tf    = target.reshape(-1)     # flatten ground truth → 1D
    total = tf.numel()             # total number of pixels

    # ── Pixel Accuracy ────────────────────────────────────────────────────────
    # How many pixels did we get right out of ALL pixels?
    pa = (pf == tf).float().mean().item()

    iou_per, acc_per, prec_per, rec_per, f1_per, freqs = [], [], [], [], [], []

    for c in range(num_classes):
        pc = pf == c   # pixels predicted as class c
        tc = tf == c   # pixels that ARE class c in ground truth

        tp    = (pc & tc).sum().float()   # correctly predicted as c
        fp    = (pc & ~tc).sum().float()  # wrongly predicted as c
        fn    = (~pc & tc).sum().float()  # missed — should be c but weren't
        union = (pc | tc).sum().float()   # all pixels involved with class c
        gt_c  = tc.sum().float()          # total GT pixels for class c

        # ── Per-class IoU ─────────────────────────────────────────────────────
        # TP / (TP + FP + FN) — how much overlap between pred and GT for class c
        iou_per.append(float('nan') if union == 0 else (tp / union).item())

        # ── Per-class Pixel Accuracy (Recall) ─────────────────────────────────
        # Of all pixels that ARE class c, how many did we correctly find?
        acc_per.append(float('nan') if gt_c == 0 else (tp / gt_c).item())

        # ── Per-class Precision ───────────────────────────────────────────────
        # Of all pixels we CALLED class c, how many actually were class c?
        pred_c = pc.sum().float()
        prec_per.append(float('nan') if pred_c == 0 else (tp / pred_c).item())

        # ── Per-class Recall ──────────────────────────────────────────────────
        # Same as per-class accuracy for segmentation
        rec_per.append(float('nan') if gt_c == 0 else (tp / gt_c).item())

        # ── Per-class F1 Score ────────────────────────────────────────────────
        # Harmonic mean of Precision and Recall
        p_val = prec_per[-1]
        r_val = rec_per[-1]
        if math.isnan(p_val) or math.isnan(r_val) or (p_val + r_val) == 0:
            f1_per.append(float('nan'))
        else:
            f1_per.append(2 * p_val * r_val / (p_val + r_val))

        # Frequency of class c (used for FWIoU)
        freqs.append((gt_c / total).item())

    # ── Mean IoU ──────────────────────────────────────────────────────────────
    # Average IoU over all classes that actually appear in this batch
    valid_iou = [v for v in iou_per if not math.isnan(v)]
    miou = float(np.mean(valid_iou)) if valid_iou else 0.0

    # ── Mean Pixel Accuracy ───────────────────────────────────────────────────
    # Average per-class accuracy — treats all classes equally regardless of size
    valid_acc = [v for v in acc_per if not math.isnan(v)]
    mpa = float(np.mean(valid_acc)) if valid_acc else 0.0

    # ── Frequency-Weighted IoU ────────────────────────────────────────────────
    # Like mIoU but bigger classes count more — closer to real-world performance
    fwiou = float(sum(freqs[c] * iou_per[c] for c in range(num_classes)
                      if not math.isnan(iou_per[c])))

    return dict(
        pa=pa, mpa=mpa, miou=miou, fwiou=fwiou,
        iou_per=iou_per, acc_per=acc_per,
        prec_per=prec_per, rec_per=rec_per, f1_per=f1_per
    )


def print_metrics_table(metrics, class_names, title='Metrics'):
    """
    Prints a clean table of all metrics to console.
    Use this after every epoch and after final evaluation.
    """
    sep = '=' * 70
    print(f'\n{sep}')
    print(f'  {title}')
    print(sep)
    # Global metrics
    print(f'  {"Pixel Accuracy":<25} {metrics["pa"]:>8.4f}   # % all pixels correct')
    print(f'  {"Mean Pixel Accuracy":<25} {metrics["mpa"]:>8.4f}   # avg per-class accuracy')
    print(f'  {"Mean IoU":<25} {metrics["miou"]:>8.4f}   # main segmentation metric')
    print(f'  {"Freq-Weighted IoU":<25} {metrics["fwiou"]:>8.4f}   # IoU weighted by class freq')
    print(sep)
    # Per-class table
    print(f'  {"Class":<16} {"IoU":>7} {"Acc":>7} {"Prec":>7} {"Rec":>7} {"F1":>7}')
    print('-' * 58)
    for c, name in enumerate(class_names):
        def fmt(v): return f'{v:.4f}' if not math.isnan(v) else '  —   '
        print(f'  {name:<16} '
              f'{fmt(metrics["iou_per"][c]):>7} '
              f'{fmt(metrics["acc_per"][c]):>7} '
              f'{fmt(metrics["prec_per"][c]):>7} '
              f'{fmt(metrics["rec_per"][c]):>7} '
              f'{fmt(metrics["f1_per"][c]):>7}')
    print(sep + '\n')

print('✓ Loss functions and metrics defined')


## 14. Dataloaders

In [ ]:
train_transform = A.Compose([
    A.Resize(config.IMG_SIZE, config.IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.3),
    A.GaussNoise(var_limit=(10,50), p=0.2),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(config.IMG_SIZE, config.IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

train_loader = val_loader = None
if config.TRAIN_PATH:
    train_dataset = LoveDADataset(config.TRAIN_PATH, train_transform)
    train_loader  = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                               shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=True)
    print(f'Train batches: {len(train_loader)}')
if config.VAL_PATH:
    val_dataset = LoveDADataset(config.VAL_PATH, val_transform)
    val_loader  = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                             shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)
    print(f'Val batches  : {len(val_loader)}')

## 15. Class Weights (sqrt median-frequency)

In [ ]:
def compute_class_weights(dataset, num_classes=7, device='cuda'):
    counts = Counter()
    for i in tqdm(range(len(dataset)), desc='Class weights'):
        _, mask = dataset[i]
        if isinstance(mask, torch.Tensor): mask = mask.numpy()
        counts.update(mask.flatten().tolist())
    total=sum(counts.values())
    freq=torch.tensor([counts.get(c,0)/total for c in range(num_classes)])
    med=torch.median(freq[freq>0])
    w=torch.zeros(num_classes)
    for c in range(num_classes):
        if freq[c]>0: w[c]=torch.sqrt(med/freq[c])
    w=w/w.sum()*num_classes
    print(f'\n{"Cls":>3}  {"Name":>12}  {"Freq":>6}  {"Weight":>7}')
    print('-'*38)
    for c,n in enumerate(LoveDADataset.CLASSES):
        print(f'  {c}  {n:>12}  {freq[c]:.4f}  {w[c]:.4f}')
    return w.to(device)

class_weights = compute_class_weights(train_dataset, config.NUM_CLASSES, device) if train_loader else None

## 16. CutMix (centre-clamped fix)

In [ ]:
def cutmix_data(images, masks, alpha=1.0, prob=0.5):
    if random.random() > prob: return images, masks, 1.0
    B,C,H,W=images.shape
    lam=np.random.beta(alpha,alpha)
    rand_idx=torch.randperm(B,device=images.device)
    cut_h=int(H*np.sqrt(1-lam)); cut_w=int(W*np.sqrt(1-lam))
    cx=random.randint(cut_w//2,max(cut_w//2+1,W-cut_w//2))
    cy=random.randint(cut_h//2,max(cut_h//2+1,H-cut_h//2))
    x1,x2=max(cx-cut_w//2,0),min(cx+cut_w//2,W)
    y1,y2=max(cy-cut_h//2,0),min(cy+cut_h//2,H)
    mi=images.clone(); mm=masks.clone()
    mi[:,:,y1:y2,x1:x2]=images[rand_idx,:,y1:y2,x1:x2]
    mm[:,  y1:y2,x1:x2]=masks [rand_idx,  y1:y2,x1:x2]
    return mi, mm, 1.0-(x2-x1)*(y2-y1)/(H*W)

print('✓ CutMix defined')

## 17. Smoke Test

In [ ]:
def smoke_test():
    print('\n'+'='*70)
    print('🧪 SMOKE TEST'.center(70))
    print('='*70+'\n')
    m = ConvNeXtVisionMambaUNet(
        backbone=config.BACKBONE, backbone_pretrained=False,
        num_classes=config.NUM_CLASSES, embed_dim=config.EMBED_DIM,
        depths=config.DEPTHS, d_state=config.D_STATE,
        d_conv=config.D_CONV, expand=config.EXPAND,
        window_size=config.WINDOW_SIZE, dropout=config.DROPOUT,
    ).to(device)

    total = sum(p.numel() for p in m.parameters())/1e6
    bb_p  = sum(p.numel() for p in m.backbone_parameters())/1e6
    new_p = sum(p.numel() for p in m.new_parameters())/1e6
    print(f'Total params        : {total:.2f}M')
    print(f'  ConvNeXt stem     : {bb_p:.2f}M  (pretrained)')
    print(f'  Vision Mamba enc  : — included in new_p')
    print(f'  Bridge+Enc+Dec    : {new_p:.2f}M  (trained)')

    x = torch.randn(2,3,config.IMG_SIZE,config.IMG_SIZE,device=device)
    t = torch.randint(0,config.NUM_CLASSES,(2,config.IMG_SIZE,config.IMG_SIZE),device=device)

    m.train()
    with autocast(device_type='cuda'):
        out  = m(x)
        loss = CombinedLoss(config.NUM_CLASSES)(out, t)
    loss.backward()
    print(f'\nMain  : {out[0].shape}  ✓')
    print(f'Aux   : {out[1].shape}  ✓')
    print(f'Loss  : {loss.item():.4f}  ✓')

    m.eval()
    with torch.no_grad(): ev = m(x)
    print(f'Eval  : {ev.shape}  ✓')
    del m; torch.cuda.empty_cache()
    print('\n✅ Smoke test passed — ready for training!\n')

smoke_test()

## 18. Train

In [ ]:
model = ConvNeXtVisionMambaUNet(
    backbone=config.BACKBONE, backbone_pretrained=config.BACKBONE_PRETRAINED,
    num_classes=config.NUM_CLASSES, embed_dim=config.EMBED_DIM,
    depths=config.DEPTHS, d_state=config.D_STATE,
    d_conv=config.D_CONV, expand=config.EXPAND,
    window_size=config.WINDOW_SIZE, dropout=config.DROPOUT,
).to(device)

total = sum(p.numel() for p in model.parameters())/1e6
bb_p  = sum(p.numel() for p in model.backbone_parameters())/1e6
print(f'Total: {total:.2f}M  (backbone: {bb_p:.2f}M)')

optimizer = optim.AdamW([
    {'params': model.backbone_parameters(), 'lr': config.LR_BACKBONE},
    {'params': model.new_parameters(),      'lr': config.LR_DECODER},
], weight_decay=config.WEIGHT_DECAY)

# CosineAnnealingWarmRestarts: warm restarts every 20 epochs for better convergence
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=1e-6)

criterion = CombinedLoss(
    num_classes=config.NUM_CLASSES, ce_weight=0.3, dice_weight=0.3,
    focal_weight=0.4, focal_gamma=2.0,
    class_weights=class_weights, aux_weight=0.4)
scaler = GradScaler()

best_miou  = 0.0
no_improve = 0
# tracks all metrics every epoch for plotting
history = {
    'train_loss': [], 'train_miou': [], 'train_pa': [], 'train_mpa': [],
    'val_loss':   [], 'val_miou':   [], 'val_pa':   [], 'val_mpa':   [],
}

model.stem.freeze()
print(f'Training {config.EPOCHS} epochs | Warmup {config.FREEZE_EPOCHS} (backbone frozen)')
print(f'LR backbone={config.LR_BACKBONE}  LR new={config.LR_DECODER}')
print(f'Early stop patience={config.PATIENCE}  Grad accum={config.ACCUM_STEPS}x\n')

for epoch in range(config.EPOCHS):

    # LR warmup / unfreeze backbone after warmup epochs
    if epoch < config.FREEZE_EPOCHS:
        frac = (epoch+1) / config.FREEZE_EPOCHS
        optimizer.param_groups[0]['lr'] = config.LR_BACKBONE * frac
        optimizer.param_groups[1]['lr'] = config.LR_DECODER * frac
    elif epoch == config.FREEZE_EPOCHS:
        model.stem.unfreeze()

    lr_b = optimizer.param_groups[0]['lr']
    lr_d = optimizer.param_groups[1]['lr']
    print(f'Epoch {epoch+1}/{config.EPOCHS}  lr_bb={lr_b:.2e}  lr_new={lr_d:.2e}')
    print('-'*65)

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    tr_loss = tr_miou = tr_pa = tr_mpa = 0.0  # reset all train metrics
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc='Train')

    for step, (images, masks) in enumerate(pbar):
        images, masks = images.to(device), masks.to(device)
        images, masks, lam = cutmix_data(images, masks, config.CUTMIX_ALPHA, config.CUTMIX_PROB)

        with autocast(device_type='cuda'):
            out  = model(images)
            loss = criterion(out, masks) / config.ACCUM_STEPS

        scaler.scale(loss).backward()

        # update weights every ACCUM_STEPS batches
        if (step+1) % config.ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if epoch >= config.FREEZE_EPOCHS:
                scheduler.step(epoch + step/len(train_loader))

        # compute all metrics for this batch — INSIDE the for step loop
        main = out[0] if isinstance(out, tuple) else out
        m = compute_all_metrics(torch.argmax(main, 1).cpu(), masks.cpu(), config.NUM_CLASSES)
        tr_loss += loss.item() * config.ACCUM_STEPS
        tr_miou += m['miou']  # Mean IoU
        tr_pa   += m['pa']    # Pixel Accuracy
        tr_mpa  += m['mpa']   # Mean Pixel Accuracy
        pbar.set_postfix({'loss': f'{loss.item()*config.ACCUM_STEPS:.4f}',
                          'mIoU': f'{m["miou"]:.4f}',
                          'PA':   f'{m["pa"]:.4f}'})

    # divide by steps — OUTSIDE the for step loop, still inside for epoch
    n = len(train_loader)
    tr_loss /= n
    tr_miou /= n
    tr_pa   /= n
    tr_mpa  /= n

    # ── Validate ──────────────────────────────────────────────────────────
    val_loss = val_miou = val_pa = val_mpa = 0.0
    if val_loader:
        model.eval()
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc='Val'):
                images, masks = images.to(device), masks.to(device)
                out  = model(images)
                loss = criterion(out, masks)
                # compute all metrics for this val batch
                m = compute_all_metrics(torch.argmax(out, 1).cpu(), masks.cpu(), config.NUM_CLASSES)
                val_loss += loss.item()
                val_miou += m['miou']  # Mean IoU
                val_pa   += m['pa']    # Pixel Accuracy
                val_mpa  += m['mpa']   # Mean Pixel Accuracy
        n = len(val_loader)
        val_loss /= n
        val_miou /= n
        val_pa   /= n
        val_mpa  /= n

    # save all metrics to history for plotting later
    history['train_loss'].append(tr_loss)
    history['train_miou'].append(tr_miou)
    history['train_pa'].append(tr_pa)
    history['train_mpa'].append(tr_mpa)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_miou)
    history['val_pa'].append(val_pa)
    history['val_mpa'].append(val_mpa)

    # print all metrics every epoch
    print(f'Train  loss={tr_loss:.4f}  mIoU={tr_miou:.4f}  PA={tr_pa:.4f}  mPA={tr_mpa:.4f}')
    if val_loader:
        print(f'Val    loss={val_loss:.4f}  mIoU={val_miou:.4f}  PA={val_pa:.4f}  mPA={val_mpa:.4f}')

    # ── Save best model ────────────────────────────────────────────────────
    if val_miou > best_miou:
        best_miou  = val_miou
        no_improve = 0
        # save best mIoU alongside all other best-epoch metrics
        torch.save({
            'epoch'      : epoch + 1,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'best_miou'  : best_miou,  # best Mean IoU achieved
            'best_pa'    : val_pa,     # Pixel Accuracy at best epoch
            'best_mpa'   : val_mpa,    # Mean Pixel Accuracy at best epoch
            'history'    : history,
        }, os.path.join(config.OUTPUT_DIR, 'best_model.pth'))
        print(f'✓ Best saved  epoch={epoch+1}  mIoU={best_miou:.4f}  PA={val_pa:.4f}  mPA={val_mpa:.4f}')
    else:
        no_improve += 1
        print(f'  No improve  ({no_improve}/{config.PATIENCE})')

    # ── Periodic checkpoint ────────────────────────────────────────────────
    if (epoch+1) % config.SAVE_FREQ == 0:
        torch.save({
            'epoch'      : epoch + 1,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'best_miou'  : best_miou,
            'history'    : history,
        }, os.path.join(config.OUTPUT_DIR, f'ckpt_epoch_{epoch+1}.pth'))
        print(f'  Checkpoint saved: epoch {epoch+1}')

    # ── Early stopping ─────────────────────────────────────────────────────
    if no_improve >= config.PATIENCE:
        print(f'\n⚡ Early stopping — no improvement for {config.PATIENCE} epochs.')
        break

print(f'\nDone — Best val mIoU: {best_miou:.4f}') 

## 19. TTA + Per-Class IoU

In [ ]:
#sir's code 
# def tta_predict(model, images):
#     model.eval()
#     with torch.no_grad():
#         p0=F.softmax(model(images),1)
#         p1=F.softmax(model(torch.flip(images,[3])),1); p1=torch.flip(p1,[3])
#         p2=F.softmax(model(torch.flip(images,[2])),1); p2=torch.flip(p2,[2])
#     return (p0+p1+p2)/3.0

# def evaluate_per_class(model, loader, device, num_classes=7):
#     model.eval()
#     inter=torch.zeros(num_classes); union=torch.zeros(num_classes)
#     with torch.no_grad():
#         for images,masks in tqdm(loader,desc='Eval TTA'):
#             images,masks=images.to(device),masks.to(device)
#             preds=torch.argmax(tta_predict(model,images),1).view(-1).cpu()
#             masks=masks.view(-1).cpu()
#             for c in range(num_classes):
#                 pc,tc=preds==c,masks==c
#                 inter[c]+=(pc&tc).sum(); union[c]+=(pc|tc).sum()
#     ious=[]
#     print(f'\n{"Cls":>3}  {"Name":>12}  {"IoU":>7}'); print('-'*28)
#     for c in range(num_classes):
#         iou=(inter[c]/(union[c]+1e-6)).item(); ious.append(iou)
#         print(f'  {c}  {LoveDADataset.CLASSES[c]:>12}  {iou:.4f}')
#     print(f'\n  mIoU (TTA): {np.mean(ious):.4f}')
#     return ious

# if val_loader:
#     ckpt = torch.load(os.path.join(config.OUTPUT_DIR,'best_model.pth'))
#     model.load_state_dict(ckpt['model_state'])
#     print(f"Loaded best model from epoch {ckpt['epoch']} with mIoU={ckpt['best_miou']:.4f}")
#     per_class_ious = evaluate_per_class(model, val_loader, device, config.NUM_CLASSES)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FINAL EVALUATION — runs after training on the best saved model
# Computes every metric on the full val set with TTA
# ─────────────────────────────────────────────────────────────────────────────

def tta_predict(model, images):
    # Test-Time Augmentation: run model 3 times with different flips
    # and average the softmax probabilities → more robust predictions
    model.eval()
    with torch.no_grad():
        p0 = F.softmax(model(images), 1)                          # original
        p1 = F.softmax(model(torch.flip(images,[3])), 1)          # horizontal flip
        p1 = torch.flip(p1,[3])                                    # flip back
        p2 = F.softmax(model(torch.flip(images,[2])), 1)          # vertical flip
        p2 = torch.flip(p2,[2])                                    # flip back
    return (p0 + p1 + p2) / 3.0                                   # average


def evaluate_full(model, loader, device, num_classes, class_names):
    """
    Runs over the entire val set and computes:
      - Pixel Accuracy (PA)
      - Mean Pixel Accuracy (mPA)
      - Per-class Accuracy
      - Mean IoU (mIoU)
      - Per-class IoU
      - Per-class Precision, Recall, F1
      - Frequency-Weighted IoU (FWIoU)
      - Confusion Matrix
      - Cohen's Kappa
    All with TTA for best accuracy.
    """
    model.eval()

    # Accumulate a 7×7 confusion matrix over all val images
    # Row = true class, Column = predicted class
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for images, masks in tqdm(loader, desc='Final Eval'):
            images = images.to(device)
            preds  = torch.argmax(tta_predict(model, images), dim=1).cpu()
            # Fill confusion matrix pixel by pixel
            np.add.at(cm, (masks.reshape(-1).numpy(),
                           preds.reshape(-1).numpy()), 1)

    # ── Derive all metrics from the confusion matrix ──────────────────────
    tp           = np.diag(cm).astype(float)         # TP per class
    fp           = cm.sum(axis=0) - tp               # FP per class
    fn           = cm.sum(axis=1) - tp               # FN per class
    total        = cm.sum()
    gt_per_class = cm.sum(axis=1).astype(float)

    # Pixel Accuracy — fraction of all pixels correctly classified
    pa = tp.sum() / (total + 1e-8)

    # Per-class Accuracy — for each class, fraction of its pixels correctly found
    acc_per = np.where(gt_per_class > 0, tp / (gt_per_class + 1e-8), np.nan)
    mpa     = float(np.nanmean(acc_per))              # Mean Pixel Accuracy

    # Per-class IoU — overlap / union for each class
    denom_iou = tp + fp + fn
    iou_per   = np.where(denom_iou > 0, tp / (denom_iou + 1e-8), np.nan)
    miou      = float(np.nanmean(iou_per))            # Mean IoU

    # Frequency-Weighted IoU — common classes contribute more
    freq  = gt_per_class / (total + 1e-8)
    fwiou = float(np.nansum(freq * np.nan_to_num(iou_per)))

    # Per-class Precision — of pixels predicted as class c, how many are correct
    pred_per_class = cm.sum(axis=0).astype(float)
    prec_per = np.where(pred_per_class > 0, tp / (pred_per_class + 1e-8), np.nan)

    # Per-class Recall — same as per-class accuracy
    rec_per = acc_per.copy()

    # Per-class F1 — harmonic mean of precision and recall
    f1_per = np.where(
        (prec_per + rec_per) > 0,
        2 * prec_per * rec_per / (prec_per + rec_per + 1e-8),
        np.nan)

    # Cohen's Kappa — agreement beyond chance (1=perfect, 0=random)
    p_observed = float(pa)
    p_expected = float(np.sum((cm.sum(axis=0)/total) * (cm.sum(axis=1)/total)))
    kappa = (p_observed - p_expected) / (1.0 - p_expected + 1e-8)

    # ── Print full results table ──────────────────────────────────────────
    sep = '=' * 70
    print(f'\n{sep}')
    print(f'  FINAL EVALUATION RESULTS (with TTA)')
    print(sep)
    print(f'  {"Pixel Accuracy (PA)":<28} {pa:>8.4f}')
    print(f'  {"Mean Pixel Accuracy (mPA)":<28} {mpa:>8.4f}')
    print(f'  {"Mean IoU (mIoU)":<28} {miou:>8.4f}')
    print(f'  {"Frequency-Weighted IoU":<28} {fwiou:>8.4f}')
    print(f'  {"Cohen\'s Kappa":<28} {kappa:>8.4f}')
    print(sep)
    print(f'  {"Class":<16} {"IoU":>7} {"Acc":>7} {"Prec":>7} {"Rec":>7} {"F1":>7}')
    print('-' * 58)
    for c, name in enumerate(class_names):
        def fmt(v): return f'{v:.4f}' if not math.isnan(v) else '  —   '
        print(f'  {name:<16} '
              f'{fmt(iou_per[c]):>7} '
              f'{fmt(acc_per[c]):>7} '
              f'{fmt(prec_per[c]):>7} '
              f'{fmt(rec_per[c]):>7} '
              f'{fmt(f1_per[c]):>7}')
    print(sep)

    # ── Plot confusion matrix (normalised + raw counts side by side) ──────
    import itertools
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ax, data, title in [
        (axes[0], cm_norm, 'Confusion Matrix (Row-Normalised)'),
        (axes[1], cm,      'Confusion Matrix (Raw Pixel Counts)'),
    ]:
        im = ax.imshow(data, cmap='Blues', vmin=0, vmax=data.max())
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ticks = np.arange(num_classes)
        ax.set_xticks(ticks); ax.set_xticklabels(class_names, rotation=45, ha='right')
        ax.set_yticks(ticks); ax.set_yticklabels(class_names)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(title, fontweight='bold')
        thresh = data.max() / 2.
        for i, j in itertools.product(range(num_classes), range(num_classes)):
            val = f'{data[i,j]:.2f}' if data.dtype == float else f'{data[i,j]:,}'
            ax.text(j, i, val, ha='center', va='center', fontsize=8,
                    color='white' if data[i,j] > thresh else 'black')
    plt.suptitle(f'Confusion Matrix  |  mIoU={miou:.4f}  PA={pa:.4f}  κ={kappa:.4f}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved → confusion_matrix.png')

    # ── Plot per-class bar chart (IoU / Accuracy / F1) ────────────────────
    def safe(arr): return [0.0 if math.isnan(v) else v for v in arr]
    x = np.arange(num_classes); w = 0.25
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w,   safe(iou_per),  w, label='IoU',      color='#4C72B0', alpha=0.85)
    ax.bar(x,       safe(acc_per),  w, label='Accuracy',  color='#DD8452', alpha=0.85)
    ax.bar(x + w,   safe(f1_per),   w, label='F1',        color='#55A868', alpha=0.85)
    ax.axhline(miou, color='#4C72B0', ls='--', lw=1.5, label=f'mIoU={miou:.4f}')
    ax.axhline(mpa,  color='#DD8452', ls='--', lw=1.5, label=f'mPA={mpa:.4f}')
    ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=20, ha='right')
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title(f'Per-Class Metrics  |  PA={pa:.4f}  mIoU={miou:.4f}  '
                 f'mPA={mpa:.4f}  FWIoU={fwiou:.4f}  κ={kappa:.4f}',
                 fontweight='bold', fontsize=10)
    ax.legend(ncol=3); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'per_class_metrics.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Saved → per_class_metrics.png')

    # ── Save all metrics to a text file for your paper ────────────────────
    txt = os.path.join(config.OUTPUT_DIR, 'final_metrics.txt')
    with open(txt, 'w') as f:
        f.write(f'Pixel Accuracy        : {pa:.4f}\n')
        f.write(f'Mean Pixel Accuracy   : {mpa:.4f}\n')
        f.write(f'Mean IoU              : {miou:.4f}\n')
        f.write(f'Frequency-Weighted IoU: {fwiou:.4f}\n')
        f.write(f'Cohen Kappa           : {kappa:.4f}\n\n')
        f.write(f'{"Class":<16} {"IoU":>7} {"Acc":>7} {"Prec":>7} {"Rec":>7} {"F1":>7}\n')
        f.write('-'*55 + '\n')
        for c, name in enumerate(class_names):
            def fmt(v): return f'{v:.4f}' if not math.isnan(v) else '  —   '
            f.write(f'{name:<16} {fmt(iou_per[c]):>7} {fmt(acc_per[c]):>7} '
                    f'{fmt(prec_per[c]):>7} {fmt(rec_per[c]):>7} {fmt(f1_per[c]):>7}\n')
    print(f'✓ Saved → final_metrics.txt')

    return dict(pa=pa, mpa=mpa, miou=miou, fwiou=fwiou, kappa=kappa,
                iou_per=iou_per, acc_per=acc_per, prec_per=prec_per,
                rec_per=rec_per, f1_per=f1_per, cm=cm)


# ── Run final evaluation on best saved model ──────────────────────────────────
if val_loader:
    ckpt = torch.load(os.path.join(config.OUTPUT_DIR, 'best_model.pth'))
    model.load_state_dict(ckpt['model_state'])
    print(f"Loaded best model — epoch {ckpt['epoch']}  mIoU={ckpt['best_miou']:.4f}")

    final = evaluate_full(model, val_loader, device,
                          config.NUM_CLASSES, LoveDADataset.CLASSES)

## 20. Training Curves & Predictions

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
ep=range(1,len(history['train_loss'])+1)
axes[0].plot(ep,history['train_loss'],'b-o',ms=3,label='Train')
axes[0].plot(ep,history['val_loss'],  'r-o',ms=3,label='Val')
axes[0].set(xlabel='Epoch',ylabel='Loss',title='Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep,history['train_miou'],'b-o',ms=3,label='Train')
axes[1].plot(ep,history['val_miou'],  'r-o',ms=3,label='Val')
axes[1].set(xlabel='Epoch',ylabel='mIoU',title='mIoU'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle(f'ConvNeXt + Vision Mamba Encoder  |  Best mIoU: {best_miou:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR,'training_curves.png'),dpi=150); plt.show()

if val_loader:
    model.eval()
    imgs,masks=next(iter(val_loader))
    imgs,masks=imgs.to(device),masks.to(device)
    with torch.no_grad(): preds=torch.argmax(tta_predict(model,imgs),1)
    mean=np.array([0.485,0.456,0.406]); std=np.array([0.229,0.224,0.225])
    n=min(3,imgs.shape[0])
    fig,axes=plt.subplots(n,3,figsize=(15,5*n))
    for i in range(n):
        img=np.clip(std*imgs[i].cpu().numpy().transpose(1,2,0)+mean,0,1)
        axes[i,0].imshow(img);                                                  axes[i,0].set_title('Input')
        axes[i,1].imshow(LoveDADataset.decode_segmap(masks[i].cpu().numpy())); axes[i,1].set_title('Ground Truth')
        axes[i,2].imshow(LoveDADataset.decode_segmap(preds[i].cpu().numpy())); axes[i,2].set_title('Pred (TTA)')
    for ax in axes.flat: ax.axis('off')
    plt.suptitle('ConvNeXt + Vision Mamba Encoder', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR,'sample_prediction.png'),dpi=150); plt.show()

## 21. Segmentation Visualization — Overlay & Per-Class Panels

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 21. Segmentation Visualization
#     • Side-by-side: Input | Ground Truth | Prediction (TTA)
#     • Colour-coded overlay of pred on original image
#     • Per-class colour legend panel
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

# LoveDA palette (matches LoveDADataset.decode_segmap)
PALETTE = np.array([
    [0,   0,   0  ],   # 0 background
    [128, 0,   0  ],   # 1 building
    [0,   128, 0  ],   # 2 road
    [128, 128, 0  ],   # 3 water
    [0,   0,   128],   # 4 barren
    [128, 0,   128],   # 5 forest
    [0,   128, 128],   # 6 agriculture
], dtype=np.uint8)

CLASS_NAMES = LoveDADataset.CLASSES   # ['background','building','road',...]
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def decode_mask(mask_np):
    """Convert H×W label map → H×W×3 RGB using PALETTE."""
    rgb = PALETTE[mask_np.astype(np.int32)]
    return rgb

def overlay(img_np, mask_np, alpha=0.45):
    """Blend colour mask onto original image."""
    colour = decode_mask(mask_np).astype(np.float32) / 255.
    return np.clip((1 - alpha) * img_np + alpha * colour, 0, 1)

def denorm(tensor):
    """CHW tensor → HWC numpy [0,1]."""
    arr = tensor.cpu().numpy().transpose(1, 2, 0)
    return np.clip(STD * arr + MEAN, 0, 1)

def make_legend(ax):
    patches = [mpatches.Patch(color=PALETTE[c]/255., label=CLASS_NAMES[c])
               for c in range(len(CLASS_NAMES))]
    ax.legend(handles=patches, loc='center', fontsize=11,
              frameon=True, ncol=1)
    ax.axis('off')
    ax.set_title('Class Legend', fontsize=12, fontweight='bold')

# ── Load best model ───────────────────────────────────────────────────────────
ckpt = torch.load(os.path.join(config.OUTPUT_DIR, 'best_model.pth'),
                  map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded best model — epoch {ckpt['epoch']}  mIoU={ckpt['best_miou']:.4f}")

# ── Grab one batch from val_loader ───────────────────────────────────────────
imgs, masks = next(iter(val_loader))
imgs, masks = imgs.to(device), masks.to(device)
with torch.no_grad():
    preds = torch.argmax(tta_predict(model, imgs), dim=1)

N_SHOW = min(4, imgs.shape[0])

# ── Figure layout: N rows × 5 cols (Input | GT | Pred | Overlay | Legend) ──
fig = plt.figure(figsize=(22, 5 * N_SHOW))
gs  = gridspec.GridSpec(N_SHOW, 5, figure=fig,
                        wspace=0.04, hspace=0.12)

col_titles = ['Input Image', 'Ground Truth', 'Prediction (TTA)',
              'Pred Overlay', 'Class Legend']

for i in range(N_SHOW):
    img_np  = denorm(imgs[i])
    gt_np   = masks[i].cpu().numpy()
    pred_np = preds[i].cpu().numpy()

    panels = [
        img_np,
        decode_mask(gt_np)   / 255.,
        decode_mask(pred_np) / 255.,
        overlay(img_np, pred_np),
    ]

    for j, (ax_data, title) in enumerate(zip(panels, col_titles)):
        ax = fig.add_subplot(gs[i, j])
        ax.imshow(ax_data)
        ax.axis('off')
        if i == 0:
            ax.set_title(title, fontsize=12, fontweight='bold', pad=6)

    # Legend column (only once per row, but looks nicer per-row)
    ax_leg = fig.add_subplot(gs[i, 4])
    make_legend(ax_leg)

plt.suptitle(
    f'ConvNeXt + Vision Mamba Encoder  —  Best mIoU: {ckpt["best_miou"]:.4f}',
    fontsize=14, fontweight='bold', y=1.01
)

out_path = os.path.join(config.OUTPUT_DIR, 'seg_visualization.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {out_path}')


## 22. Confusion Matrix (Normalised & Raw)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 22. Confusion Matrix
#     • Accumulates predictions over entire val set (TTA)
#     • Plots both normalised (row %) and raw count matrices
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
import itertools

def build_confusion_matrix(model, loader, device, num_classes=7):
    """Return (num_classes × num_classes) confusion matrix over full val set."""
    model.eval()
    all_gt, all_pred = [], []
    with torch.no_grad():
        for images, masks in tqdm(loader, desc='Building CM'):
            images = images.to(device)
            preds  = torch.argmax(tta_predict(model, images), dim=1)
            gt_flat   = masks.view(-1).cpu().numpy()
            pred_flat = preds.view(-1).cpu().numpy()
            # ignore pixels outside [0, num_classes-1] if any
            valid = (gt_flat >= 0) & (gt_flat < num_classes)
            all_gt.append(gt_flat[valid])
            all_pred.append(pred_flat[valid])
    all_gt   = np.concatenate(all_gt)
    all_pred = np.concatenate(all_pred)
    cm = confusion_matrix(all_gt, all_pred, labels=list(range(num_classes)))
    return cm

def plot_confusion_matrix(cm, class_names, title='Confusion Matrix',
                          normalise=True, figsize=(10, 8), cmap='Blues'):
    if normalise:
        row_sums = cm.sum(axis=1, keepdims=True).astype(float)
        cm_plot  = np.where(row_sums == 0, 0, cm / (row_sums + 1e-8))
        fmt, vmax = '.2f', 1.0
        bar_label = 'Row-normalised proportion'
    else:
        cm_plot  = cm
        fmt, vmax = 'd', cm.max()
        bar_label = 'Pixel count'

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm_plot, interpolation='nearest', cmap=cmap,
                   vmin=0, vmax=vmax)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(bar_label, fontsize=11)

    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks); ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(tick_marks); ax.set_yticklabels(class_names, fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label',      fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)

    # Annotate cells
    thresh = cm_plot.max() / 2.
    for i, j in itertools.product(range(cm_plot.shape[0]),
                                   range(cm_plot.shape[1])):
        val = (f'{cm_plot[i,j]:.2f}' if normalise
               else f'{cm_plot[i,j]:,}')
        ax.text(j, i, val, ha='center', va='center', fontsize=9,
                color='white' if cm_plot[i, j] > thresh else 'black')

    plt.tight_layout()
    return fig

# ── Build CM over full val set ────────────────────────────────────────────────
print('Building confusion matrix over validation set (TTA)…')
cm = build_confusion_matrix(model, val_loader, device, num_classes=config.NUM_CLASSES)

# ── Plot normalised ───────────────────────────────────────────────────────────
fig_norm = plot_confusion_matrix(
    cm, CLASS_NAMES,
    title=f'Confusion Matrix (Row-Normalised) — mIoU {ckpt["best_miou"]:.4f}',
    normalise=True, figsize=(10, 8)
)
norm_path = os.path.join(config.OUTPUT_DIR, 'confusion_matrix_norm.png')
fig_norm.savefig(norm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {norm_path}')

# ── Plot raw counts ───────────────────────────────────────────────────────────
fig_raw = plot_confusion_matrix(
    cm, CLASS_NAMES,
    title='Confusion Matrix (Raw Pixel Counts)',
    normalise=False, figsize=(10, 8), cmap='YlOrRd'
)
raw_path = os.path.join(config.OUTPUT_DIR, 'confusion_matrix_raw.png')
fig_raw.savefig(raw_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {raw_path}')

# ── Per-class precision / recall / F1 from CM ────────────────────────────────
print('\n{:<14} {:>9} {:>9} {:>9} {:>9}'.format(
      'Class', 'Precision', 'Recall', 'F1', 'IoU'))
print('-' * 50)
for c in range(config.NUM_CLASSES):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    iou  = tp / (tp + fp + fn + 1e-8)
    print(f'{CLASS_NAMES[c]:<14} {prec:>9.4f} {rec:>9.4f} {f1:>9.4f} {iou:>9.4f}')
miou = sum(cm[c,c]/(cm[c,:].sum()+cm[:,c].sum()-cm[c,c]+1e-8)
           for c in range(config.NUM_CLASSES)) / config.NUM_CLASSES
print(f'\nmIoU (from CM): {miou:.4f}')


## 23. Quantitative Metrics Table (mIoU / mAcc / OA / mF1)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 23. Full Quantitative Metrics
#     mIoU | Per-class IoU | mAcc | OA | mF1 | Kappa
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import cohen_kappa_score

def compute_all_metrics(model, loader, device, num_classes=7):
    model.eval()
    inter  = torch.zeros(num_classes)
    union  = torch.zeros(num_classes)
    tp_cls = torch.zeros(num_classes)   # for per-class accuracy
    tot    = torch.zeros(num_classes)   # total GT pixels per class
    all_gt, all_pred = [], []

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Metrics"):
            images, masks = images.to(device), masks.to(device)
            preds = torch.argmax(tta_predict(model, images), dim=1)
            p = preds.view(-1).cpu()
            g = masks.view(-1).cpu()
            all_gt.append(g.numpy())
            all_pred.append(p.numpy())
            for c in range(num_classes):
                pc, gc = p == c, g == c
                inter[c] += (pc & gc).sum()
                union[c] += (pc | gc).sum()
                tp_cls[c]+= (pc & gc).sum()
                tot[c]   += gc.sum()

    all_gt   = np.concatenate(all_gt)
    all_pred = np.concatenate(all_pred)

    iou_per  = (inter / (union + 1e-8)).numpy()
    miou     = iou_per.mean()
    acc_per  = (tp_cls / (tot + 1e-8)).numpy()
    macc     = acc_per.mean()
    oa       = (all_gt == all_pred).mean()
    f1_per   = np.array([
        2*iou_per[c] / (1 + iou_per[c] + 1e-8) for c in range(num_classes)
    ])
    mf1      = f1_per.mean()
    kappa    = cohen_kappa_score(
                   all_gt[:5_000_000], all_pred[:5_000_000])   # sample for speed

    return dict(iou_per=iou_per, miou=miou, acc_per=acc_per,
                macc=macc, oa=oa, f1_per=f1_per, mf1=mf1, kappa=kappa)

metrics = compute_all_metrics(model, val_loader, device, config.NUM_CLASSES)

# ── Pretty print ──────────────────────────────────────────────────────────────
print("=" * 62)
print(f"  {'Metric':<20} {'Value':>10}")
print("-" * 62)
print(f"  {'mIoU':<20} {metrics['miou']:>10.4f}")
print(f"  {'mAcc':<20} {metrics['macc']:>10.4f}")
print(f"  {'OA':<20} {metrics['oa']:>10.4f}")
print(f"  {'mF1':<20} {metrics['mf1']:>10.4f}")
print(f"  {'Kappa':<20} {metrics['kappa']:>10.4f}")
print("=" * 62)
print()
print(f"  {'Class':<14} {'IoU':>7} {'Acc':>7} {'F1':>7}")
print("-" * 40)
for c in range(config.NUM_CLASSES):
    print(f"  {LoveDADataset.CLASSES[c]:<14} "
          f"{metrics['iou_per'][c]:>7.4f} "
          f"{metrics['acc_per'][c]:>7.4f} "
          f"{metrics['f1_per'][c]:>7.4f}")
print("=" * 62)

# ── Bar chart — per-class IoU, Acc, F1 ───────────────────────────────────────
x = np.arange(config.NUM_CLASSES)
w = 0.26
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w,   metrics['iou_per'], w, label='IoU',  color='steelblue',  alpha=0.85)
ax.bar(x,       metrics['acc_per'], w, label='Acc',  color='darkorange',  alpha=0.85)
ax.bar(x + w,   metrics['f1_per'],  w, label='F1',   color='seagreen',    alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(LoveDADataset.CLASSES, rotation=20, ha='right')
ax.set_ylim(0, 1.05); ax.set_ylabel('Score'); ax.legend()
ax.set_title(f"Per-Class Metrics  |  mIoU={metrics['miou']:.4f}  "
             f"mAcc={metrics['macc']:.4f}  OA={metrics['oa']:.4f}  "
             f"mF1={metrics['mf1']:.4f}  κ={metrics['kappa']:.4f}",
             fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, 'per_class_metrics.png')
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f"✓ Saved → {out}")


## 24. Model Efficiency — FLOPs, Parameters & Inference Speed

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 24. Model Efficiency
#     FLOPs (GFLOPs), Parameters (M), Throughput (FPS), Latency (ms/img)
# ─────────────────────────────────────────────────────────────────────────────
!pip install thop -q

from thop import profile, clever_format
import time

model.eval()
dummy = torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE).to(device)

# FLOPs & Params
with torch.no_grad():
    flops, params = profile(model, inputs=(dummy,), verbose=False)
flops_f, params_f = clever_format([flops, params], "%.3f")

# Warm-up
with torch.no_grad():
    for _ in range(20):
        _ = model(dummy)
torch.cuda.synchronize()

# Latency — 200 runs
times = []
with torch.no_grad():
    for _ in range(200):
        t0 = time.perf_counter()
        _ = model(dummy)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

times  = np.array(times[10:])   # drop first 10 warm-up
lat_mean = times.mean()
lat_std  = times.std()
fps      = 1000.0 / lat_mean

# Batch throughput (bs=8)
dummy8 = torch.randn(8, 3, config.IMG_SIZE, config.IMG_SIZE).to(device)
batch_times = []
with torch.no_grad():
    for _ in range(50):
        t0 = time.perf_counter()
        _ = model(dummy8)
        torch.cuda.synchronize()
        batch_times.append((time.perf_counter() - t0) * 1000)
batch_fps = 8 * 1000.0 / np.mean(batch_times[5:])

# GPU memory
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _ = model(dummy8)
mem_mb = torch.cuda.max_memory_allocated() / 1024**2

print("=" * 50)
print("  MODEL EFFICIENCY REPORT")
print("=" * 50)
print(f"  GFLOPs        : {flops/1e9:.2f}")
print(f"  Parameters    : {params/1e6:.2f} M")
print(f"  Latency       : {lat_mean:.1f} ± {lat_std:.1f} ms/img  (bs=1)")
print(f"  FPS (bs=1)    : {fps:.1f}")
print(f"  Throughput    : {batch_fps:.1f} img/s (bs=8)")
print(f"  GPU Mem (bs=8): {mem_mb:.0f} MB")
print("=" * 50)

# ── Summary bar chart ─────────────────────────────────────────────────────────
labels  = ["GFLOPs", "Params (M)", "Latency (ms)", "FPS"]
values  = [flops/1e9, params/1e6, lat_mean, fps]
colors  = ["#4C72B0","#DD8452","#55A868","#C44E52"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, lbl, val, col in zip(axes, labels, values, colors):
    ax.bar([lbl], [val], color=col, alpha=0.85, width=0.4)
    ax.text(0, val * 1.02, f"{val:.2f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.set_ylim(0, val * 1.3)
    ax.set_title(lbl, fontsize=12, fontweight="bold")
    ax.set_xticks([])
    ax.grid(axis="y", alpha=0.3)
plt.suptitle("ConvNeXt + Vision Mamba — Model Efficiency", fontsize=13, fontweight="bold")
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "model_efficiency.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")


## 25. Urban vs Rural Domain Split Analysis (LoveDA)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 25. Urban vs Rural mIoU breakdown
#     LoveDA val contains Urban/ and Rural/ subfolders — evaluate separately
# ─────────────────────────────────────────────────────────────────────────────
import torchvision.transforms as T

def get_domain_loader(root, domain, transform, batch_size=4):
    """Build a loader for a single domain (Urban or Rural)."""
    img_dir  = os.path.join(root, "Val", domain, "images_png")
    mask_dir = os.path.join(root, "Val", domain, "masks_png")
    if not os.path.isdir(img_dir):
        print(f"  ⚠  {img_dir} not found — skipping {domain}")
        return None
    ds = LoveDADataset(img_dir, mask_dir, transform=transform)
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=2, pin_memory=True)

val_tf = val_transform   # re-use notebook's val_transform

results = {}
for domain in ["Urban", "Rural"]:
    loader = get_domain_loader(config.DATA_ROOT, domain, val_tf,
                               batch_size=config.BATCH_SIZE)
    if loader is None:
        continue
    print(f"\n── {domain} ({len(loader.dataset)} samples) ──")
    inter = torch.zeros(config.NUM_CLASSES)
    union = torch.zeros(config.NUM_CLASSES)
    model.eval()
    with torch.no_grad():
        for imgs, msks in tqdm(loader, desc=domain):
            imgs, msks = imgs.to(device), msks.to(device)
            preds = torch.argmax(tta_predict(model, imgs), dim=1)
            p, g  = preds.view(-1).cpu(), msks.view(-1).cpu()
            for c in range(config.NUM_CLASSES):
                pc, gc = p==c, g==c
                inter[c] += (pc & gc).sum()
                union[c] += (pc | gc).sum()
    iou_per = (inter / (union + 1e-8)).numpy()
    miou    = iou_per.mean()
    results[domain] = {"miou": miou, "iou_per": iou_per}
    print(f"  mIoU ({domain}): {miou:.4f}")
    for c in range(config.NUM_CLASSES):
        print(f"    {LoveDADataset.CLASSES[c]:<14} {iou_per[c]:.4f}")

# ── Side-by-side bar chart ────────────────────────────────────────────────────
if len(results) == 2:
    x  = np.arange(config.NUM_CLASSES)
    w  = 0.35
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - w/2, results["Urban"]["iou_per"], w,
           label=f"Urban  (mIoU={results['Urban']['miou']:.4f})",
           color="#4C72B0", alpha=0.85)
    ax.bar(x + w/2, results["Rural"]["iou_per"], w,
           label=f"Rural  (mIoU={results['Rural']['miou']:.4f})",
           color="#DD8452", alpha=0.85)
    ax.axhline(results["Urban"]["miou"], color="#4C72B0", ls="--", lw=1.2, alpha=0.6)
    ax.axhline(results["Rural"]["miou"], color="#DD8452", ls="--", lw=1.2, alpha=0.6)
    ax.set_xticks(x); ax.set_xticklabels(LoveDADataset.CLASSES, rotation=20, ha="right")
    ax.set_ylim(0, 1.05); ax.set_ylabel("IoU"); ax.legend(fontsize=11)
    ax.set_title("Urban vs Rural — Per-Class IoU (LoveDA Val)",
                 fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    out = os.path.join(config.OUTPUT_DIR, "urban_vs_rural_iou.png")
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"\n✓ Saved → {out}")

    # Domain gap
    gap = abs(results["Urban"]["miou"] - results["Rural"]["miou"])
    print(f"\n  Domain gap (|Urban mIoU − Rural mIoU|): {gap:.4f}")


## 26. Grad-CAM & Encoder Feature Maps

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 26. Grad-CAM on decoder output + Encoder scale feature maps
# ─────────────────────────────────────────────────────────────────────────────
!pip install grad-cam -q

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denorm(t):
    return np.clip(STD * t.cpu().numpy().transpose(1,2,0) + MEAN, 0, 1)

model.eval()
imgs, masks = next(iter(val_loader))
imgs = imgs.to(device)

# ── Grad-CAM: target the final decoder conv before head ───────────────────────
# Adjust target_layer to your model's actual last conv name
try:
    target_layer = [model.final_head[0]]   # first conv in final_head
except:
    # fallback: last conv of decoder
    target_layer = [list(model.decoder.children())[-1]]

N_SHOW = min(3, imgs.shape[0])
fig, axes = plt.subplots(N_SHOW, config.NUM_CLASSES + 1, figsize=(3*(config.NUM_CLASSES+1), 4*N_SHOW))

for i in range(N_SHOW):
    img_np = denorm(imgs[i]).astype(np.float32)
    axes[i, 0].imshow(img_np); axes[i, 0].set_title("Input", fontsize=9); axes[i, 0].axis("off")
    for cls_idx in range(config.NUM_CLASSES):
        targets = [SemanticSegmentationTarget(cls_idx,
                       torch.ones(1, config.IMG_SIZE, config.IMG_SIZE))]
        with GradCAM(model=model, target_layers=target_layer) as cam:
            grayscale = cam(input_tensor=imgs[i:i+1], targets=targets)[0]
        cam_img = show_cam_on_image(img_np, grayscale, use_rgb=True)
        axes[i, cls_idx+1].imshow(cam_img)
        axes[i, cls_idx+1].set_title(LoveDADataset.CLASSES[cls_idx], fontsize=9)
        axes[i, cls_idx+1].axis("off")

plt.suptitle("Grad-CAM — Per-Class Activation Maps", fontsize=13, fontweight="bold")
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "gradcam.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")

# ── Encoder feature maps (4 scales via forward hooks) ────────────────────────
feature_maps = {}
hooks = []

def make_hook(name):
    def hook(module, inp, out):
        feature_maps[name] = out.detach().cpu()
    return hook

# Hook onto bridge outputs (one per scale)
for i, layer in enumerate(model.bridge.proj):
    hooks.append(layer.register_forward_hook(make_hook(f"scale_{i}")))

with torch.no_grad():
    _ = model(imgs[0:1])

for h in hooks:
    h.remove()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i in range(4):
    key = f"scale_{i}"
    if key not in feature_maps:
        axes[i].axis("off"); continue
    fm = feature_maps[key]       # (1, C, H, W) or (1, HW, C)
    if fm.dim() == 3:            # token form → reshape
        B, L, C = fm.shape
        H = W = int(L**0.5)
        fm = fm.reshape(B, H, W, C).permute(0, 3, 1, 2)
    avg = fm[0].mean(0).numpy()  # average over channels
    im  = axes[i].imshow(avg, cmap="viridis")
    axes[i].set_title(f"Bridge Scale {i}  {tuple(fm.shape[1:])}", fontsize=10, fontweight="bold")
    axes[i].axis("off")
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.suptitle("Encoder Bridge — Mean Feature Map per Scale", fontsize=13, fontweight="bold")
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "encoder_feature_maps.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")


## 27. t-SNE Feature Embedding

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 27. t-SNE of bottleneck features coloured by ground-truth class
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.manifold import TSNE

TSNE_SAMPLES = 8000   # pixels to embed (keep manageable)
PALETTE_NORM  = PALETTE / 255.

# ── Extract bottleneck features via hook ──────────────────────────────────────
bottleneck_feats = {}

def bn_hook(module, inp, out):
    bottleneck_feats["feat"] = out.detach().cpu()

# Hook last bottleneck Mamba block
hook_bn = model.bottleneck[-1].register_forward_hook(bn_hook)

all_feats, all_labels = [], []
model.eval()
n_batches = 4   # use 4 batches to get enough pixels

with torch.no_grad():
    for k, (imgs_b, msks_b) in enumerate(val_loader):
        if k >= n_batches: break
        imgs_b = imgs_b.to(device)
        _ = model(imgs_b)
        fm = bottleneck_feats["feat"]       # (B, C, H, W) or (B, HW, C)
        if fm.dim() == 3:
            B, L, C = fm.shape
            H = W = int(L**0.5)
            fm = fm.reshape(B, H, W, C).permute(0, 3, 1, 2)
        B, C, H, W = fm.shape
        # Downsample masks to match fm spatial size
        msks_down = F.interpolate(
            msks_b.float().unsqueeze(1), size=(H, W),
            mode="nearest").squeeze(1).long()
        feat_flat  = fm.permute(0,2,3,1).reshape(-1, C).numpy()
        label_flat = msks_down.reshape(-1).numpy()
        all_feats.append(feat_flat)
        all_labels.append(label_flat)

hook_bn.remove()

all_feats  = np.concatenate(all_feats,  axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Subsample
idx = np.random.choice(len(all_feats), min(TSNE_SAMPLES, len(all_feats)), replace=False)
feats_sub  = all_feats[idx]
labels_sub = all_labels[idx]

print(f"Running t-SNE on {len(feats_sub)} points …")
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000,
            random_state=42, n_jobs=-1)
emb  = tsne.fit_transform(feats_sub)

# ── Plot ──────────────────────────────────────────────────────────────────────
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(10, 8))
for c in range(config.NUM_CLASSES):
    mask = labels_sub == c
    if mask.sum() == 0: continue
    ax.scatter(emb[mask, 0], emb[mask, 1],
               c=[PALETTE_NORM[c]], s=4, alpha=0.6, label=LoveDADataset.CLASSES[c])
ax.legend(markerscale=4, fontsize=10, loc="best")
ax.set_title("t-SNE of Bottleneck Features (coloured by GT class)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
ax.grid(alpha=0.2)
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "tsne_features.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")


## 28. Failure Case Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 28. Failure Case Analysis
#     Ranks validation samples by worst mIoU and shows the hardest cases
# ─────────────────────────────────────────────────────────────────────────────
from torch.utils.data import DataLoader, Subset

def sample_miou(pred_np, gt_np, num_classes=7):
    """Compute mIoU for a single image pair."""
    ious = []
    for c in range(num_classes):
        inter = ((pred_np == c) & (gt_np == c)).sum()
        union = ((pred_np == c) | (gt_np == c)).sum()
        if union == 0: continue
        ious.append(inter / (union + 1e-8))
    return float(np.mean(ious)) if ious else 0.0

model.eval()
scores = []   # (miou, img_tensor, gt, pred)

# Evaluate individual samples
single_loader = DataLoader(val_loader.dataset, batch_size=1,
                           shuffle=False, num_workers=2)
with torch.no_grad():
    for imgs_s, msks_s in tqdm(single_loader, desc="Scoring samples"):
        imgs_s = imgs_s.to(device)
        pred_s = torch.argmax(tta_predict(model, imgs_s), dim=1)[0].cpu().numpy()
        gt_s   = msks_s[0].numpy()
        sc     = sample_miou(pred_s, gt_s, config.NUM_CLASSES)
        scores.append((sc, imgs_s[0].cpu(), msks_s[0], torch.tensor(pred_s)))

# Sort by mIoU ascending → worst cases first
scores.sort(key=lambda x: x[0])
N_FAIL = 4

MEAN_A = np.array([0.485,0.456,0.406]); STD_A = np.array([0.229,0.224,0.225])
def denorm(t):
    return np.clip(STD_A * t.numpy().transpose(1,2,0) + MEAN_A, 0, 1)

fig, axes = plt.subplots(N_FAIL, 3, figsize=(14, 5*N_FAIL))
for i, (sc, img_t, gt_t, pred_t) in enumerate(scores[:N_FAIL]):
    img_np  = denorm(img_t).astype(np.float32)
    gt_np   = gt_t.numpy()
    pred_np = pred_t.numpy()
    axes[i,0].imshow(img_np)
    axes[i,0].set_title(f"Input  (mIoU={sc:.4f})", fontsize=10); axes[i,0].axis("off")
    axes[i,1].imshow(decode_mask(gt_np)/255.)
    axes[i,1].set_title("Ground Truth", fontsize=10); axes[i,1].axis("off")
    axes[i,2].imshow(decode_mask(pred_np)/255.)
    axes[i,2].set_title("Prediction (TTA)", fontsize=10); axes[i,2].axis("off")

plt.suptitle(f"Failure Cases — {N_FAIL} Worst Validation Samples",
             fontsize=13, fontweight="bold")
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "failure_cases.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")

# ── Also show best cases for comparison ──────────────────────────────────────
fig2, axes2 = plt.subplots(N_FAIL, 3, figsize=(14, 5*N_FAIL))
for i, (sc, img_t, gt_t, pred_t) in enumerate(scores[-N_FAIL:][::-1]):
    img_np  = denorm(img_t).astype(np.float32)
    axes2[i,0].imshow(img_np)
    axes2[i,0].set_title(f"Input  (mIoU={sc:.4f})", fontsize=10); axes2[i,0].axis("off")
    axes2[i,1].imshow(decode_mask(gt_t.numpy())/255.)
    axes2[i,1].set_title("Ground Truth", fontsize=10); axes2[i,1].axis("off")
    axes2[i,2].imshow(decode_mask(pred_t.numpy())/255.)
    axes2[i,2].set_title("Prediction (TTA)", fontsize=10); axes2[i,2].axis("off")

plt.suptitle(f"Best Cases — {N_FAIL} Top Validation Samples",
             fontsize=13, fontweight="bold")
plt.tight_layout()
out2 = os.path.join(config.OUTPUT_DIR, "best_cases.png")
fig2.savefig(out2, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out2}")


## 29. Boundary Quality Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 29. Boundary Quality
#     • Extracts GT and Pred boundaries via morphological erosion
#     • Computes Boundary F1 (BF) score per class
#     • Visualises boundary overlay on sample images
# ─────────────────────────────────────────────────────────────────────────────
from scipy.ndimage import binary_erosion

def get_boundary(mask, ignore_val=255, border_px=2):
    boundary = np.zeros_like(mask, dtype=bool)
    for c in np.unique(mask):
        if c == ignore_val: continue
        region   = mask == c
        eroded   = binary_erosion(region, iterations=border_px)
        boundary |= (region ^ eroded)
    return boundary

def boundary_f1(pred, gt, num_classes, border_px=2, threshold=0.008):
    """Boundary F1 score per class (relaxed boundary matching)."""
    gt_bd   = get_boundary(gt,   border_px=border_px)
    pred_bd = get_boundary(pred, border_px=border_px)
    bf_per  = []
    for c in range(num_classes):
        gt_c   = (gt == c) & gt_bd
        pred_c = (pred == c) & pred_bd
        if gt_c.sum() == 0 and pred_c.sum() == 0:
            bf_per.append(1.0); continue
        if gt_c.sum() == 0 or pred_c.sum() == 0:
            bf_per.append(0.0); continue
        prec = (gt_c & pred_c).sum() / (pred_c.sum() + 1e-8)
        rec  = (gt_c & pred_c).sum() / (gt_c.sum()  + 1e-8)
        bf   = 2*prec*rec / (prec + rec + 1e-8)
        bf_per.append(float(bf))
    return bf_per

model.eval()
all_bf = np.zeros(config.NUM_CLASSES)
n_batches_bf = 10
count = 0

with torch.no_grad():
    for k, (imgs_b, msks_b) in enumerate(tqdm(val_loader, desc="Boundary F1")):
        if k >= n_batches_bf: break
        imgs_b = imgs_b.to(device)
        preds_b = torch.argmax(tta_predict(model, imgs_b), dim=1).cpu().numpy()
        for pred_np, gt_np in zip(preds_b, msks_b.numpy()):
            all_bf += np.array(boundary_f1(pred_np, gt_np, config.NUM_CLASSES))
            count  += 1

all_bf /= (count + 1e-8)
mean_bf = all_bf.mean()

print("=" * 45)
print("  BOUNDARY F1 SCORES")
print("=" * 45)
for c in range(config.NUM_CLASSES):
    print(f"  {LoveDADataset.CLASSES[c]:<14}  BF={all_bf[c]:.4f}")
print(f"  {'Mean BF':<14}  BF={mean_bf:.4f}")
print("=" * 45)

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(LoveDADataset.CLASSES, all_bf, color="mediumpurple", alpha=0.85)
ax.axhline(mean_bf, color="red", ls="--", lw=1.5, label=f"Mean BF={mean_bf:.4f}")
ax.set_ylim(0, 1.05); ax.set_ylabel("Boundary F1")
ax.set_title("Per-Class Boundary F1 Score", fontsize=13, fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "boundary_f1.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")

# ── Boundary overlay on sample images ────────────────────────────────────────
MEAN_A = np.array([0.485,0.456,0.406]); STD_A = np.array([0.229,0.224,0.225])
def denorm(t):
    return np.clip(STD_A * t.cpu().numpy().transpose(1,2,0) + MEAN_A, 0, 1)

imgs_b, msks_b = next(iter(val_loader))
imgs_b = imgs_b.to(device)
with torch.no_grad():
    preds_b = torch.argmax(tta_predict(model, imgs_b), dim=1).cpu()

N_SHOW = min(3, imgs_b.shape[0])
fig, axes = plt.subplots(N_SHOW, 3, figsize=(14, 5*N_SHOW))
for i in range(N_SHOW):
    img_np  = denorm(imgs_b[i]).astype(np.float32)
    gt_np   = msks_b[i].numpy()
    pred_np = preds_b[i].numpy()
    gt_bd   = get_boundary(gt_np).astype(np.float32)
    pred_bd = get_boundary(pred_np).astype(np.float32)

    # Overlay: GT boundary = green, Pred boundary = red
    overlay_img = img_np.copy()
    overlay_img[gt_bd   > 0] = [0.0, 1.0, 0.0]
    overlay_img[pred_bd > 0] = [1.0, 0.2, 0.2]

    axes[i,0].imshow(img_np);    axes[i,0].set_title("Input",          fontsize=10); axes[i,0].axis("off")
    axes[i,1].imshow(decode_mask(gt_np)/255.); axes[i,1].set_title("Ground Truth", fontsize=10); axes[i,1].axis("off")
    axes[i,2].imshow(overlay_img); axes[i,2].set_title("Boundary Overlay\n(green=GT  red=Pred)", fontsize=9); axes[i,2].axis("off")

plt.suptitle("Boundary Quality — GT vs Prediction Edges",
             fontsize=13, fontweight="bold")
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "boundary_overlay.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")


## 30. Learning Rate Schedule Curve

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 30. LR Schedule Curve
#     Reconstructs the backbone and new-head LR schedules from history
# ─────────────────────────────────────────────────────────────────────────────
if "lr_bb" in history and "lr_new" in history:
    ep = range(1, len(history["lr_bb"]) + 1)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ep, history["lr_bb"],  "b-o", ms=3, label="LR backbone")
    ax.plot(ep, history["lr_new"], "r-o", ms=3, label="LR new head")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate")
    ax.set_title("Learning Rate Schedule (Warmup + Cosine Annealing)",
                 fontsize=12, fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = os.path.join(config.OUTPUT_DIR, "lr_schedule.png")
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved → {out}")
else:
    # Reconstruct from config if not logged
    warmup = config.WARMUP_EPOCHS
    total  = config.EPOCHS
    lr_bb_list, lr_new_list = [], []
    for ep in range(1, total+1):
        if ep <= warmup:
            lr_bb_list.append(config.LR_BACKBONE * ep / warmup)
            lr_new_list.append(config.LR_NEW * ep / warmup)
        else:
            t = (ep - warmup) / (total - warmup)
            cos = 0.5 * (1 + np.cos(np.pi * t))
            lr_bb_list.append(config.LR_BACKBONE * cos)
            lr_new_list.append(config.LR_NEW * cos)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, total+1), lr_bb_list,  "b-", lw=2, label="LR backbone")
    ax.plot(range(1, total+1), lr_new_list, "r-", lw=2, label="LR new head")
    ax.axvline(warmup, color="gray", ls="--", lw=1.2, label=f"Warmup end (ep {warmup})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate")
    ax.set_title("Reconstructed LR Schedule (Warmup + Cosine Annealing)",
                 fontsize=12, fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = os.path.join(config.OUTPUT_DIR, "lr_schedule.png")
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
    print(f"✓ Saved → {out}")


## 31. Statistical Significance — Multi-Seed Report

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 31. Statistical Significance
#     Run inference with 3 different dropout seeds → report mean ± std mIoU
#     (Full multi-seed training is done offline; this shows inference variance)
# ─────────────────────────────────────────────────────────────────────────────
from scipy import stats

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def quick_miou(model, loader, device, num_classes=7, max_batches=30):
    """Fast mIoU over max_batches batches (no TTA for speed)."""
    model.eval()
    inter = torch.zeros(num_classes); union = torch.zeros(num_classes)
    with torch.no_grad():
        for k, (imgs_b, msks_b) in enumerate(loader):
            if k >= max_batches: break
            imgs_b = imgs_b.to(device)
            preds  = torch.argmax(
                F.softmax(model(imgs_b), dim=1), dim=1).cpu()
            for c in range(num_classes):
                pc, gc = preds.view(-1)==c, msks_b.view(-1)==c
                inter[c] += (pc & gc).sum(); union[c] += (pc | gc).sum()
    return (inter / (union + 1e-8)).mean().item()

SEEDS = [42, 123, 2024]
seed_mious = []

for seed in SEEDS:
    set_seed(seed)
    # Enable MC-Dropout by activating dropout layers
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout): m.train()
    sc = quick_miou(model, val_loader, device, config.NUM_CLASSES)
    seed_mious.append(sc)
    print(f"  Seed {seed:4d} → mIoU = {sc:.4f}")

# Restore eval mode
model.eval()

mean_miou = np.mean(seed_mious)
std_miou  = np.std(seed_mious)
print(f"\n  mIoU = {mean_miou:.4f} ± {std_miou:.4f}  (n={len(SEEDS)} seeds)")

# One-sample t-test against a hypothetical baseline mIoU
baseline = 0.52   # replace with your best baseline
t_stat, p_val = stats.ttest_1samp(seed_mious, baseline)
print(f"  t-test vs baseline ({baseline}): t={t_stat:.3f}  p={p_val:.4f}")
sig = "✓ Significant (p<0.05)" if p_val < 0.05 else "✗ Not significant"
print(f"  {sig}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([f"Seed {s}" for s in SEEDS], seed_mious,
       color=["#4C72B0","#DD8452","#55A868"], alpha=0.85)
ax.axhline(mean_miou, color="red",  ls="--", lw=1.5, label=f"Mean={mean_miou:.4f}")
ax.axhline(baseline,  color="gray", ls=":",  lw=1.5, label=f"Baseline={baseline}")
ax.fill_between(range(len(SEEDS)),
                mean_miou - std_miou, mean_miou + std_miou,
                alpha=0.15, color="red", label=f"±1σ ({std_miou:.4f})")
ax.set_ylim(min(seed_mious)*0.97, max(seed_mious)*1.03)
ax.set_ylabel("mIoU"); ax.legend(fontsize=9)
ax.set_title(f"Multi-Seed mIoU  ({mean_miou:.4f} ± {std_miou:.4f})",
             fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
out = os.path.join(config.OUTPUT_DIR, "statistical_significance.png")
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"✓ Saved → {out}")


## 32. Save All Outputs

In [ ]:
!zip -r outputs.zip outputs
print("✓ outputs.zip ready for download")
print("\nOutputs include:")
for fname in sorted(os.listdir(config.OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(config.OUTPUT_DIR, fname))
    print(f"  {fname:<45} {size/1024:.1f} KB")
